# Lab notebook — how hard will a new test item be?

**Project:** Levante QA (quality checks for child assessments)  
**Folder:** `tools/vlm-panel/`  
**In one sentence:** We ask AI models to take the same items kids take, then guess how hard a **new** item will be *before* we have child data.

This is a lab log: add dated notes; do not silently rewrite old results. Numbers live in files under `out/`.

**Start here if you are not a psychometrician.** We are *not* publishing final difficulties into the live test. We are testing whether AI rankings help researchers sort “this looks easy / hard / broken.”

### The story so far

1. **Picture + sentence tasks (TROG, vocab, stories):** AI rankings often line up with kids about as well as a careful human first guess (ρ around 0.64–0.75). English vocab hybrid is `z`+AoA (held-out ρ **0.749**). First German / Colombian-Spanish 12-cell smokes are weaker (ρ **0.49** / **0.58**). The “age kids learn this word” table we use (Kuperman) is **English-only** — other languages reuse the English lemma via `item_uid`. A tight translation sieve — only flag when **our guessed difficulty** (`d_est`) moves by 3+ vs English — yields **7 words**, **5** later matching a 3+ kids IRT swing (same easier/harder sign). Useful for sorting / blank priors, **not** as the official CAT `d`.
2. **Word reading (SWR):** Best so far is ranking real words with “age kids learn this word,” and made-up words with spelling patterns. Held-out match ρ ≈ 0.59. Still a draft list.
3. **Puzzles (matrix, shape rotation) and math:** Weak or unproven. Do not treat AI order as kid order.
4. **Writing brand-new items:** Sentence reading (SRE) has a published recipe. We have not run that ourselves.

### Glossary (say these out loud)

| We write | Plain meaning |
|----------|----------------|
| **Item** | One question / word / puzzle in the bank |
| **Task** | A whole game (TROG = grammar pictures; vocab = pick the picture; SWR = real vs made-up word) |
| **CAT** | Adaptive test: the computer picks the next item from a difficulty number |
| **Bank `d` or `b`** | Official “how hard?” from thousands of kids. **Higher = harder** on the bank we use. New items start blank |
| **`difficulty` (vocab locale)** | Kids IRT written into the locale bank (higher = harder). Live CAT still prefers leftover **`d`** when both exist. ±5 = placeholder, not IRT |
| **`b_hat` / `d_est`** | **Our guess** of that official number (a draft, not a replacement) |
| **`b_proxy`** | Raw AI ranking score (right order-ish, wrong units) |
| **Panel / VLM** | Several AI “respondents” (vision-language models) taking the item |
| **`p` / pass rate** | Fraction who get it right (0 = nobody, 1 = everybody) |
| **`p_pred`** | That fraction **stretched toward real kids** |
| **ρ (rho)** | Did two rankings agree? **0 = no better than chance, 1 = same order.** About 0.7 is “useful sort”; 0.2 is not |
| **MAE** | Average miss, in the same units as the thing we guessed |
| **GO / LEAN-GO / NO-GO** | Use this for *research ranking* / promising / do not use |
| **Pseudo** | Made-up word (`ggnoi`), not a real English word |
| **AoA** | Age of acquisition: typical age people say they learned a word. Our table (Kuperman 2012) is **English**. German/Spanish fits reuse the English lemma or fill the median |
| **Ceiling / BROKEN** | Everyone gets it right / the item looks unsolvable (often a bad key) |
| **Ruler B (`d_est` Δ)** | Locale guessed difficulty minus English guessed difficulty. **+ = we think the translation is harder; − = easier.** Empty bank `difficulty` is missing, not zero |

---

## Table of contents

1. [Question & framing](#1-question--framing)
2. [Tooling inventory](#2-tooling-inventory)
3. [Experiment track: age gradients](#experiment-track-age-gradients)
4. [Prompt history (from git)](#prompt-history-from-git)
5. [Chronology of work](#3-chronology-of-work)
6. [Key quantitative results](#4-key-quantitative-results)
7. [Conceptual learnings](#5-conceptual-learnings)
8. [Open questions / next](#6-open-questions--next)
9. [Reload live artifacts](#7-reload-live-artifacts)
10. [Appendix A — Ma et al. 2025 (BEA) inference RC AIG](#appendix-a--ma-et-al-2025-bea-inference-rc-aig)

## 1. Question & framing

### Goal

When someone writes a **new question** (or a translation), we want a first guess of **how many children will get it right** and **where it sits on the easy→hard list** — before any child has taken it.

That matters most for **adaptive tests (CAT)**: grammar pictures (TROG), pattern puzzles (matrix), shape rotation, same-different. Those tests pick the next item using an official hardness number (`d` or `b`).

**What this is for (2026-08-10):** help **human researchers** sort and sanity-check items. **Not** uploading numbers into the live item bank (GCS/Crowdin).

### What “success” means

- We mainly check **TROG** and **vocab**, because we already know how kids did.
- Success = “our easy→hard list matches the kids’ list” (ρ) and “we are not wildly off on percent correct” (MAE). Success is **not** “ship a patch to production.”
- English runs must use locale **`en-US`**. Bare `en` breaks audio and the AI sees a black screen.
### Two different “difficulty” targets

| Target | Meaning | Use |
|--------|---------|-----|
| Predicted kid percent correct (`p_pred_child`) | “About what % of children get this right?” | Quick triage |
| Bank hardness (`d` or `b`) | Official difficulty from a huge kid sample. **Higher = harder** | Which item the adaptive test serves next |
| Research export (`item_params`) | Another difficulty file for papers | Report only — sometimes **flipped** (high = easy). Do not mix with bank `d` |

**Where official `d` comes from:** a statistical model fit on many children’s answers, then stored in the cloud bank. A brand-new item has **no** official `d` until we collect child data.

**Working rule (2026-08-07):** for a new adaptive item we may fill a **blank** `d` with our guess (`d_est`) as a starting prior, then **refit after real kids**.
 See §5 “CAT initial-`d` prior.”

## 2. Tooling inventory

This is the **toolbox** (scripts and config), not the findings. Skip it unless you need to re-run something. “Panel” = AI models taking items; `d_est` / `b_hat` = our guessed difficulty.

| Piece | Path | Role |
|-------|------|------|
| Panel runner | `run_panel.mjs` + `panel_grid*.json` | Ungated VLM respondents × ages × models |
| Analyze | `analyze.mjs` | `p_vlm`, human join, calibrator → `p_pred_child` |
| Bench calibrator | `fit_bench_calibrator.mjs` | Fit on levante-bench trials |
| Hybrid `d_est` | `estimate_difficulty.mjs` | Map `p_pred` (+ TROG tags / vocab **AoA**) → bank scale. Vocab fit: kids IRT `difficulty` (`vocabIrtD`) |
| Apply `d_est` prior | `apply_d_est_prior.mjs` | Fill **blank** bank `d`/`difficulty` only; skip `BROKEN` / `known_issues.json`; draft CSV (no GCS upload). Tasks: trog, vocab, stories, **matrix** |
| Vocab locale banks | `corpora/vocab/vocab-item-bank-*.csv` | Registered `-prod` copies. Fit = `difficulty` (skip ±5). Apply-prior / live CAT = `d` else `difficulty` |
| Vocab pairwise (offline) | `eval_vocab_pairwise.mjs` | Text-only “A or B harder?” → Bradley–Terry `bt` vs kids IRT |
| Vocab AoA probe | `eval_vocab_aoa.mjs` | Kuperman 2012 **English** age-of-acquisition vs kids IRT / Zipf / blanks |
| ICC `d_icc` | `fit_icc_difficulty.mjs` | Fit Rasch-with-guessing from age→θ panel trials |
| Age gradient check | `eval_age_gradient.mjs` | `med_p` by age, item spreads |
| TROG prompts | `cypress/support/agents/prompts/trogPrompts.ts` | Age-conditional system + `trogUserText` (imported by agent) |
| TROG agent | `cypress/support/agents/trogVlmAgent.ts` | Cypress decide loop; re-exports prompts |
| Matrix grid | `panel_grid_matrix_smoke.json` | EN smoke; `task=matrix`; identity=`stimulusAlt` (`joinByItemId`) |
| Paper-ladder backtest (Stories) | `out/REPORT_stories_paper_ladder_backtest.md` | Aggregate levante-bench open ToM runs → ρ vs human IRT |
| Vocab paper-ladder backtest | `out/REPORT_vocab_paper_ladder_backtest.md` | Same zoo on `vocab.csv`; **NO-GO** vs Cypress |
| Vocab soft-score analyze | `analyze_vocab_soft.mjs` (+ `panel_grid_vocab_soft_v3.json`) | Retro YES\|NO / pre-randomize digit vs bank `d`; future logs `knowsWord` |
| Vocab graded v4 | `panel_grid_vocab_graded_v4.json` + `analyze_vocab_graded.mjs` + `QA_VOCAB_PROMPT=v4` | HIGH\|MED\|LOW smoke; `grid.env` via `run_panel.mjs` |
| Persona | `cypress/support/persona/childPersona.ts` | Age/θ preamble + TROG mastery cues |
| Learnings | `LEARNINGS.md`, `RESULTS.md` | Operator docs |
| SRE gen/sim paper notes | `out/NOTES_zelikman_emnlp2023_sre.md` | Zelikman et al. EMNLP 2023 — GPT item gen + student-response simulator + OT parallel forms |
| SWR prompts / agent | `cypress/support/agents/prompts/swrPrompts.ts`, `swrVlmAgent.ts` | `QA_SWR_PROMPT=v1\|v2\|v2strict\|v3`; optional `QA_SWR_AOA` / `QA_SWR_CHILD_PLAY` |
| SWR panel grid | `panel_grid_swr.json` + `run_roar_panel.mjs` | EN/DE ATM cells; `--run-suffix` |
| SWR `b_est` | `eval_swr_b_est.mjs` (`--aoa-blend auto`) | Graded pChild + Kuperman blend → `b_proxy` vs human `b` |
| SWR / vocab AoA data | `data/aoa_kuperman.csv`, `lib/aoa.mjs` | Kuperman 2012 **English** norms only. de/es vocab lookup is English lemma via `item_uid` (same 154 hits / median 7.32 as EN); no German or Spanish AoA table in-repo |
| SWR prompt bake-off / matched | `eval_swr_prompt_bakeoff.mjs`, `eval_swr_aoa_*.mjs` | Offline HML / `hml_s` / HARDNESS; `--source bank`; AoA helpers |
| SWR draft ranking export | `export_swr_draft_ranking.mjs`, `export_swr_bank_bakeoff_ranking.mjs` | Live a10 + bank age-avg CSVs (`out/swr_draft_rank_*.csv`) |
| SWR pseudo separate-track | `eval_swr_pseudo_features.mjs` | Real = AoA `b_proxy`; pseudo = VLM p + ortho (bank bigrams / length) |
| TROG AIG (specs + pictures) | `aig_trog_specs.mjs`, `aig_trog_images.mjs`, `aig_trog_score.mjs` | Write EN passive / X-but-not-Y items, 2×2 clipart, frozen v4 `d_est` prior |
| PA ES VLM vs human | `eval_pa_es_human_baseline.mjs`, `eval_pa_es_prompt_bakeoff.mjs`, `eval_pa_es_vlm_vs_human.mjs` | Bogotá trials + roar-pa ES corpus; text-only Gemini |

**Panel formula (ungated):** VLM answers real UI → empirical `p_vlm` → monotonic calibrator → `p_pred_child`.

**Do not use** `QA_PERSONA_GATE=irt` for new items without a prior (needs item `d`; blanks collapse to age-mean `fallbackP`). Prefer ungated panel → `d_est`, then `QA_SIM_D_EST_PRIOR` / draft bank fill for missing `d` only.

## Experiment track: age gradients

**In plain English:** If the AI is acting like children, **older “AI kids” should get the same item right more often**. Adaptive tests hide that in real life (older kids get harder items). Our panel shows every age the same items on purpose.

**Role:** a supporting check for “can we back out official hardness from age curves?” — not a separate research program.

**Why it mattered:** The AI’s percent-correct barely rose with age (spread ~0.10 vs ~0.21 expected). That is why the curve-fitting hardness (`d_icc`) failed (ρ ≈ 0.05 — almost no match to the official list).

**What we tried**

| Approach | Outcome |
|----------|---------|
| Soft “act like a 6-year-old” persona | Already failed for age curves (`LEARNINGS.md`) — do not revive |
| Same adult TROG grammar checklist at all ages | Flattens curves (age-6 parses like an adult) |
| Age-conditional checklist (≤8 light / ≥10 full) + TROG mastery cues | Mini-eval **GO**: Δ med_p(13−6) 0.030 → **0.071**; item mean spread 0.086 → **0.152**; full-panel MAE p_pred **0.059** |

**How we measure**

1. Primary: respondent `med_p` by age → Δ(a12 − a6) via `eval_age_gradient.mjs` (was a13)
2. Item `max(p)−min(p)` across ages (mean/median spread)
3. Guardrail: full-panel MAE `p_pred` ≤ ~0.09
4. Downstream: `fit_icc_difficulty.mjs` ρ vs −p_pred / multivar `d_est`

**What “good” accuracy looks like (VLM ≠ child)**

Replay prints running `acc=correct/scored` per cell. That is **panel accuracy**, not child pass rates.

| Signal | Expectation |
|--------|-------------|
| EN age-6 VLM cell (end of bank) | Typically **~0.75–0.86** (historical median ≈ **0.81**); stronger cells can finish **~0.94+** |
| Mid-run (e.g. 63/70 ≈ 0.90) | **OK** — early TROG items are easier; totals often drift down by the end |
| Real age-6 kids (θ ≈ −2.0, n≈99) | Much lower than VLM totals — soft “act like 6” never closed that gap |
| What we care about | Age-6 ends **below** older ages (Δ med_p), item age-spreads, and MAE `p_pred` ≤ ~0.09 — not child-like raw accuracy |

Grid ages are now **`[6, 8, 10, 12]`** (was 13; θ₁₂ ≈ −0.22, n=104). Prefer capture-once + offline replay (`run_panel.mjs --capture-assets` then replay).

**Artifacts:** `out/age_grad_baseline_pre.json`, `out/age_grad_after.json`, `out/age_eval_gonogo.md`, `panel_grid_trog_age_eval.json`

**Status (2026-08-07):** GO for fuller EN force recollect with age-conditional prompts; ICC still weak until ages 8/10/12 are refreshed under the new grid. Append dated results under Chronology (§3) as this track continues.


## Prompt history (from git)

**In plain English:** These are the instruction rewrites we gave the AI, with version numbers. Later sections cite “v3” or “v4” — this table is what that means.

Canonical TROG prompt text now lives in
 [`cypress/support/agents/prompts/trogPrompts.ts`](../../cypress/support/agents/prompts/trogPrompts.ts) (extracted from `trogVlmAgent.ts` so wording can be versioned and reviewed without agent plumbing). Child persona preamble remains [`cypress/support/persona/persona_template.txt`](../../cypress/support/persona/persona_template.txt); TROG age-band mastery cues are appended in `childPersona.ts`.

Reconstructed from `git log` / `git show` on `trogVlmAgent.ts` (and persona template):

| Date | Commit | Prompt change |
|------|--------|----------------|
| 2026-05-31 | `3a232e5` *add trog* | **v0 baseline:** hear sentence → pick picture; short grammar reminder (word order, who/whom, negation, prepositions, clauses). Digit-only reply. No checklist, no `trogUserText`. |
| 2026-05-31 | `5bdefa1` / `d945985` | Same TROG system prompt. Age persona + IRT θ preamble added (`persona_template.txt`, `QA_PERSONA_ABILITY=irt`) — prepended in `cypress.config`, not in the TROG agent file. |
| 2026-07-30 | `027133a` | Child Twins panel plumbing; TROG `SYSTEM_PROMPT` still v0. |
| 2026-08-03 | `757cc01` *update model matrix, add results* | **v1 checklist:** five silent checks (agent/patient, negation scope, spatial, comparative, relative clauses). First **`trogUserText`** structure hints (negation, despite, spatial, size, chase/push). |
| 2026-08-06 | `4bcf684` *trog & vocab difficulties* | **v2 checklist:** passives explicit; spatial list + under/beneath; comparatives “named pair only”; embeddings example; **item 6** contrast connectives (despite/although/however/instead). Richer `trogUserText` (passive `by`, embeddings regex). Used for full EN force recollect → `d_est` ρ 0.532 → 0.637. |
| 2026-08-06 | *(working tree / lab)* age-conditional | **v3:** `SYSTEM_PROMPT_CHECKLIST` (v2) for age ≥10; **`SYSTEM_PROMPT_YOUNG`** (no checklist) for age ≤8; young runs skip structure hints. TROG mastery cues in persona by age band. Mini-eval GO (Δ med_p 0.030 → 0.071). |
| 2026-08-07 | *(lab)* | Prompts moved to `prompts/trogPrompts.ts`; agent re-exports for compatibility. |
| 2026-08-07 | *(lab / working tree)* | **v4 checklist + hints (ages ≥10 only):** HEAD-NOUN + tighter contrast + no agent-reversal on modifier `-ing`. Smoke 16 cells: duck ~0.13→**0.63**; despite ~0.20→**0.31**; Δ(12−6)=**0.101**; car/truck **0.00** (regression). |
| 2026-08-07 | *(lab / working tree)* | **v4.1 head-noun split:** `that/who` → keep required participants; participial → reject other noun doing Z. Remeasure 16 cells: car/truck still **0.00**; duck 0.63→**0.44**; despite **0.38**. **NO-GO** on car/truck wording. |
| 2026-08-07 | *(lab / working tree)* | **v4.2 two-noun relative tip:** user hint only when `the X that/who the Y`; drop broad that/who hint. Participial unchanged. Smoke: car/truck **0.06** (still BROKEN); duck **0.56**; ρ **0.53**. **NO-GO**. |
| 2026-08-07 | *(lab / working tree)* | **Freeze = v4 wording** (best duck) + `trog_preploc_car_truck_follow_drive` added to `known_issues.json`. Stop relative rewrites; de/es refresh on frozen prompt. |

**Design tension (still active):** checklist improves absolute TROG calibration (models were too hard on structure) but, applied at every age, flattens age/θ curves needed for ICC `d_icc`. v3/v4 keep the young/light split; v4 only deepens structure rules for older personas.

**Related persona note:** Soft “act like a 6-year-old” alone did not produce age curves (`LEARNINGS.md`). Operational mastery + age-conditional *task* scaffolding is the path we kept.

When editing prompts: change `trogPrompts.ts`, force-recollect affected grid cells, append a dated row here + Chronology metrics.

### Vocab prompts

Canonical text: [`cypress/support/agents/prompts/vocabPrompts.ts`](../../cypress/support/agents/prompts/vocabPrompts.ts) (extracted from `vocabVlmAgent.ts`). Age-conditional young vs checklist; agent wires via `vocabSystemPrompt` / `vocabUserText`.

| Date | Version / commit | What changed |
|------|------------------|--------------|
| (pre-2026-08) | inline in `vocabVlmAgent.ts` | Single static system prompt: ordinary concrete match; digit 1–4 only. |
| 2026-08-09 | **v1** | Extract `vocabPrompts.ts`. **Age split** (≤8 young / >8 checklist). Checklist adds school-age ordinary meaning + anti-stretch / no encyclopedic rescue for rare words. Young: short ordinary-match, no thesaurus. `vocabUserText` restates the word. Smoke: `panel_grid_vocab_prompt_eval.json` (**must `--live`**). **NO-GO** (CEILING ↑). |
| 2026-08-09 | **v2** | Stronger uncommon-word rule (“you may not know it — do not force a match”). **Age vocabulary limit** injected from persona age (`Vocabulary limit: … typical N-year-old`). Still operational scaffolding, not soft roleplay. Builders: `buildSystemPromptYoung` / `buildSystemPromptChecklist`. |
| 2026-08-10 | **v3** | Reply `DIGIT YES\|NO` (know word at age?). Agent **`applyKnowsWordPolicy`**: NO → uniform random 1–4. Temp ladder via `temperatures[]` in `run_panel.mjs`. Grid: `panel_grid_vocab_ability_v3.json` (current models only). |
| 2026-08-11 | **v4** *(smoke; `QA_VOCAB_PROMPT=v4`)* | Reply `DIGIT HIGH\|MED\|LOW`. LOW→randomize; soft 1/0.5/0.25. Grid: `panel_grid_vocab_graded_v4.json` (8 cells). Default stays v3 until GO. Live smoke → `REPORT_vocab_graded_v4.md`. |
| 2026-08-12 | **v4 remmeasure** | Full smoke **NO-GO**: 7/8 cells; −p_vlm **0.592** > −p_graded **0.530** vs bank `d`. Keep v3 default. |
| 2026-08-27 | **pairwise-a** *(offline; not Cypress)* | Text-only “which word is harder? A or B” for a typical 8-year-old. Wins → Bradley–Terry `bt`. `eval_vocab_pairwise.mjs`. |

### SWR prompts

Canonical text: [`cypress/support/agents/prompts/swrPrompts.ts`](../../cypress/support/agents/prompts/swrPrompts.ts). Env: `QA_SWR_PROMPT=v1|v2|v2strict|v3`, optional `QA_SWR_CHILD_PLAY`.

| Date | Version | What changed |
|------|---------|--------------|
| 2026-08-13 | **v1** | LEFT / RIGHT adult lexicality only. Live EN `langfix` ρ(b_proxy,b)≈**0.18**. |
| 2026-08-13 | **v2** | `REAL\|PSEUDO` + `HIGH\|MED\|LOW` child-success. Soft 1/0.5/0.25 → `p_child`. Text-only when DOM word known. Live `v2smoke4`: ρ(b_proxy,b)≈**0.38**. |
| 2026-08-14 | **v3** | Bake-off winner: `REAL\|PSEUDO` + HARDNESS **1–5** → `p_child=(6−h)/5`. Offline `h15_age_avg` held-out ρ≈**0.516**. Live age-6 `v3smoke2`: ρ(b_proxy,b)≈**−0.20** (plumbing GO / metrics weak). |
| 2026-08-15 | **v2strict** | Same HML; stricter HIGH (default MED). Bank age-avg ρ≈**0.56**, ceil 5%. |

### PA ES prompts (offline text-only)

Canonical text: `eval_pa_es_prompt_bakeoff.mjs` (not Cypress). FSM/LSM stim/goal/foils; ages 6 and 10 → age-avg. No audio.

| Date | Version | What changed |
|------|---------|--------------|
| 2026-08-15 | **pa-es v1** | Spanish PA child difficulty. `hml_s` (HIGH/MED/LOW, prefer MED) and `h15` (HARDNESS 1–5). Winner vs Bogotá: `h15` ρ=**0.485**. |


## 3. Chronology of work

**How to read this log:** each dated block is one experiment. **Verdict GO** = use for research ranking; **NO-GO** = do not. ρ = “did our easy→hard list match the kids?” (0 = chance, 1 = perfect). Paths under `out/` have the full tables. Older entries keep the original shorthand; the [glossary](#glossary-say-these-out-loud) defines the symbols.

### 2026-08 — Early framing

- Confirmed levante-qa **consumes** GCS bank `d` / persona θ; it does not fit IRT for new items.
- Best existing tool for unscored items: **VLM panel** → calibrator.
- GCS bank `d` and Redivis `item_params` are **different scales** (TROG overlap Pearson ≈ −0.39 historically).

### Affine → hybrid `d_est`

1. **v1** `estimate_difficulty.mjs`: affine `d_est = α + β·z` from `p_pred_child`.
   - TROG Spearman vs bank `d` weak (~0.24).
   - Vocab ranking stronger (~0.62); affine cannot beat pass-rate ranking ceiling.
2. **v2 hybrid:** ridge + Huber IRLS; features = `z` + TROG construction tags (`tagResidual`) / vocab Zipf.
   - TROG held-out **ρ_multivar ≈ 0.532** vs p-only ceiling **≈ 0.284** (beats ceiling).
   - Vocab: ranking ~0.61; Zipf helps MAE more than ranking.
3. Strengthened TROG prompts (`trogVlmAgent.ts`: passive, comparative, despite/however, embeddings).

### Limited prompt-eval recollect (8 cells)

- Grid: `panel_grid_trog_prompt_eval.json` (ages 8/10).
- Bug: `QA_LANGUAGE=en` → audio 404; fixed to **`en-US`**.
- n=8 too small/noisy to claim prompt lift (ceiling worsened 0.284 → 0.211 in that snapshot).

### Full EN TROG force recollect (32 cells)

- `panel_grid.json`, `--lang en-US --force` (~6h; muted/paused mid-run for Zoom, then resumed).
- Post: `post_en_full_recollect.sh` → analyze → fit calibrator → `estimate_difficulty` vs `d_est_trog_en_baseline_full.json`.

**After full recollect (2026-08-06):**

| Metric | baseline | after | Δ |
|--------|----------|-------|---|
| Spearman multivar | 0.532 | **0.637** | +0.105 |
| −p_pred ceiling | 0.284 | **0.471** | +0.187 |
| MAE multivar | 0.886 | **0.822** | −0.063 |
| MAE p_pred vs human | 0.076 | **0.063** | −0.013 |

Artifacts: `out/d_est_trog_en_report.md`, `out/trog_en_pred_after.json`.

### Idea: match new-item `p_pred` to similar `p_child`

- Reasonable for **triage** (same as p-only ranking).
- Insufficient for bank-scale CAT `d` when bank `d` ≉ reorder of pass rates (TROG Spearman(d_bank, −p_human) ≈ 0.43).

### Idea: variety of θs → ICC `d_icc`

- Implemented `fit_icc_difficulty.mjs`:  
  `P(correct|θ) = c + (1−c)·sigmoid(θ − d_icc)`  
  with θ from `age_task_ability.json`, then CV affine link to bank `d`.
- **Result (pre age-conditional prompts):** linked ρ ≈ **0.054**; raw ρ ≈ 0.21; −p_pred on same anchors ≈ 0.25; multivar `d_est` ≈ **0.637**.
- Diagnosis: **flat VLM×age curves** (median item age spread ~0.10 vs ~0.21 Rasch-expected given bank `d`). Fixed panel (all ages see same items) *should* show larger per-item age gradients than CAT overall accuracy.

### Age-conditional TROG child-likeness (2026-08-06 → 08-07)

**Hypothesis:** same adult grammar checklist at every age flattens age curves. Soft “act like a 6-year-old” already failed (`LEARNINGS.md`) — instead make **task prompt** age-conditional.

**Changes:**

1. `trogVlmAgent.ts`: age ≤8 → light prompt, no checklist / no structure `trogUserText` hints; age ≥10 → keep checklist.
2. `childPersona.ts`: TROG mastery cues by age band (operational, not cute roleplay).
3. Eval grid: `panel_grid_trog_age_eval.json` (2 models × ages 6/13 × 2 reps = 8 cells).

**Age-eval results:**

| | before | after |
|--|--------|-------|
| med_p age 6 | 0.894 | 0.818 |
| med_p age 13 | 0.924 | 0.889 |
| **Δ med_p(13−6)** | 0.030 | **0.071** |
| item mean age spread | 0.086 | **0.152** |
| Full-panel MAE p_pred | — | **0.059** (≤0.09) |

- One cell failed: `35flashlite_a6_r1`.
- **Verdict: GO** toward fuller EN force recollect (`out/age_eval_gonogo.md`).
- ICC on mixed panel still weak (ρ_cv ≈ 0.06) — only a6/a13 refreshed.

### 2026-08-07 — Known-issue triage + prompt v4 (postmod / despite)

**Context:** After capture/replay + `analyze.mjs --human-source=bench`, EN review listed three BROKEN items. Grid ages are now `[6,8,10,12]`.

#### Corpus / triage change

| Item | Action | Why |
|------|--------|-----|
| `trog_conjcoord_say_sunny_however_rain` | Added to [`known_issues.json`](known_issues.json); **suppressed from `review_*.csv` / `review_xlang_*.csv`** | Known broken for everyone (human p≈0.04, panel p≈0.02). Still on `screen_*.csv` with `KNOWN:` reason — not an actionable panel finding. |
| `trog_postmod_duck_following_turtle_walking` | Kept on review | See below |
| `trog_disjunctive_despite_noise_she_focus` | Kept on review | See below |

Wiring: `analyze.mjs` loads `known_issues.json` per task; known UIDs stay on the full screen, drop out of review triage.

#### Why the other two were BROKEN (not mis-keys)

| Item | p_vlm | p_human | Dominant VLM error |
|------|-------|---------|-------------------|
| duck … following … walking | 0.13 | 0.49 | Picks both animals on bridge (`duck-turtle-on-bridge`) instead of head-only (`duck-on-bridge`) — **relative / participial postmodifier** failure |
| despite noise … reading | 0.20 | 0.42 | Picks `…-writes` instead of `…-reads` — **main-clause activity** under a concessive |

#### Prompt change (v4)

General construction rules in `trogPrompts.ts` (no item names): HEAD-NOUN rule + tighter contrast-connective wording + stop agent-reversal hint on modifier `-ing`. Ages ≤8 unchanged. See Prompt history table.

**Run:** not yet — waiting on force replay / limited smoke before full 32-cell.

**Verdict:** triage corpus cleaned; prompt hypothesis ready to test.

**Next:** force replay (or smoke on relative_clause + disjunctive items) → re-analyze → compare duck/despite flags and MAE p_pred / age Δ.

### 2026-08-07 — Prompt v4 smoke replay (16 cells)

**Hypothesis:** HEAD-NOUN + tighter contrast rules lift duck/despite without item-specific coaching; age Δ stays healthy.

**Change:** v4 prompts (already in `trogPrompts.ts`). Grid: [`panel_grid_trog_v4_smoke.json`](panel_grid_trog_v4_smoke.json) (2 models × ages 6/8/10/12 × **2** repeats = 16).

**Run:** `node tools/vlm-panel/run_panel.mjs --force --replay --lang en-US --grid tools/vlm-panel/panel_grid_trog_v4_smoke.json` (~63 min). Log: `out/logs/v4_smoke_replay.log`.

**Metrics** (smoke cells only; pooled item correctness):

| Metric | Pre-v4 (prior panel) | v4 smoke |
|--------|----------------------|----------|
| duck postmod p | ~0.13 | **0.625** (10/16) |
| despite/noise p | ~0.20 | **0.31** (5/16) |
| Δ med_p(12−6) | ~0.07–0.09 | **0.101** |

**Verdict:** **GO** on head-noun (duck). Despite only a small lift (still near chance) — contrast rule weak or construction still hard. Age gradient OK.

**Analyze** (`--run-id-re` smoke 16 cells only — unfiltered disk mix still shows duck BROKEN from old r3/r4):

| Item | flag | p_vlm | p_human |
|------|------|-------|---------|
| duck postmod | **OK** | 0.625 | 0.487 |
| despite/noise | **HARD** | 0.313 | 0.419 |
| sunny (known) | BROKEN (suppressed) | 0.000 | 0.040 |
| car truck follow (new concern) | **BROKEN** | 0.000 | 0.617 |

EN smoke screen: B2/H14/C60; ρ difficulty **0.57**; CV MAE p_pred **0.083**. Artifacts: `out/report.md`, `out/screen_en.csv`, `out/review_en.csv` (from filtered run).

**Next:** full repeats=4 force vs despite iteration; inspect `trog_preploc_car_truck_follow_drive` (relative + reverse agent — head-noun family).

### 2026-08-07 — Prompt v4.1 (head-noun: keep required participants)

**Hypothesis:** v4’s “reject other noun doing Z” was too broad for `that/who` relatives; models dropped required participants (car/truck → car-only).

**Change:** `trogPrompts.ts` — split head-noun hints:
- **`that/who`:** outer action on head; **still include** nouns the relative requires (no head-only that drops them).
- **Participial postmod:** unchanged intent — reject other noun also doing the same main action (duck case).

**Evidence (v4 smoke):** car/truck keyed `car-truck-into-tunnel`; 12/16 picked `car-into-tunnel`; p 0.31→**0.00**.

**Run:** `panel_grid_trog_v4_smoke.json` force replay (~1h). Log: `out/logs/v4_1_smoke_replay.log`.

**Metrics (16 cells):**

| Item | v4 | v4.1 |
|------|----|------|
| duck | 0.625 | **0.438** (partial regress) |
| despite | 0.31 | **0.375** |
| car/truck | 0.00 | **0.00** (still 14/16 `car-into-tunnel`) |
| Δ med_p(12−6) | 0.101 | **0.101** |

**Verdict:** **NO-GO** on car/truck wording — models still drop the required truck. Duck softer than v4. Need a different relative-clause strategy (or accept as HARD and don’t over-fit).

**Next:** rethink `that/who` hint (maybe: resolve who-did-what inside the relative *and* keep all clause participants); avoid trading duck for car.

### 2026-08-07 — Prompt v4.2 (two-noun relative tip)

**Hypothesis:** Broad that/who “keep participants” hints don’t fix car/truck and hurt duck; a tip that fires only on `the X that/who the Y` will lift car/truck without trading duck.

**Change:** `trogPrompts.ts` — user hint only when two overt nouns in that/who relative (“both must appear; reject head-only”); drop broad that/who user hint; participial path unchanged; checklist bullet tightened to the two-noun case.

**Run:** `panel_grid_trog_v4_smoke.json` force replay. Log: `out/logs/v4_2_smoke_replay.log`. Analyze: `--run-id-re 'panel_trog_en_(35flashlite|36flash)_a(6|8|10|12)_r[12]$'`.

**Metrics (16 cells):**

| Item / metric | v4 | v4.1 | v4.2 |
|---------------|----|------|------|
| duck | 0.625 | 0.438 | **0.563** HARD |
| despite | 0.31 | 0.375 | **0.375** HARD |
| car/truck | 0.00 | 0.00 | **0.063** BROKEN (still mostly `car-into-tunnel`) |
| ρ difficulty | 0.57 | 0.59 | **0.53** |
| MAE p_pred | 0.083 | 0.08 | **0.08** |
| Δ med_p(12−6) | 0.101 | 0.101 | **0.096** |

**Verdict:** **NO-GO**. Car/truck essentially still broken; duck partially recovered vs v4.1 but below v4; ρ slipped. Stop iterating general relative wording — accept car/truck as known/HARD or try a non-prompt lever.

**Next:** mark car/truck known-issue *or* freeze prompt near best duck (v4) and full-force; don’t chase another head-noun rewrite.

### 2026-08-07 — Freeze EN TROG + parallel de/es + vocab

**Decision:** Stop relative-clause prompt chasing. Freeze prompt at **v4** (best duck smoke). Add `trog_preploc_car_truck_follow_drive` to [`known_issues.json`](known_issues.json) (still on `screen_*.csv`).

**Parallel work started:**
1. **de/es TROG force** on frozen v4 via `run_langs_trog.mjs` + [`panel_grid_trog_xlang_limited.json`](panel_grid_trog_xlang_limited.json) (16 cells/lang). Log: `out/logs/xlang_de_es_force.log`. Analyze/`review_xlang_*.csv` after finish.
2. **Vocab refresh** (existing panels, no recollect): analyze + `estimate_difficulty` → ρ_multivar **0.613**, −p_pred ceiling **0.661**, MAE **1.399** (unchanged vs prior). Artifacts: `out/report_vocab.md`, `out/d_est_vocab_en_*`.

**Verdict:** freeze landed; xlang running; vocab metrics stable (prompting not the lever).

**Next:** when de/es done → analyze bench + triage `review_xlang_{de,es}.csv`; EN full-force only if we want bank `d_est` refresh on frozen v4.

### 2026-08-07 — de/es TROG force complete

**Run:** `run_langs_trog.mjs` + `panel_grid_trog_xlang_limited.json` force de-DE then es-CO (16 cells each, live). ~3.7h. Log: `out/logs/xlang_de_es_force.log`. `worst_cell_exit=0`.

**Analyze** (`--human-source=bench`, full disk mix + fresh 3.x cells):

| Lang | resp | flags B/H/C | ρ diff | MAE p_pred |
|------|------|-------------|--------|------------|
| en | 88 | 4/12/14 | 0.65 | 0.07 |
| de | 69 | 2/13/16 | 0.64 | 0.08 |
| es | 64 | 2/13/12 | 0.62 | 0.08 |

**Triage:** duck still BROKEN on de (p≈0.11) and es (p≈0.22). Xlang strong |Δ|≥0.25: **de 1** (`trog_abovebelow_square_below_star` Δ≈−0.27); **es 0**. Artifacts: `out/review_{de,es}.csv`, `out/review_xlang_{de,es}.csv`.

**Verdict:** de/es panels refreshed on frozen v4; few translation-delta candidates (es clean; one spatial DE).

**Next:** optional EN full-force for `d_est`; spot-check DE square/star item.

### 2026-08-07 — Decision: accept hybrid `d_est` (not model-θ / ICC) for new-item difficulty

- **What:** For new / unscored items, primary bank-scale estimate = hybrid `d_est` from `p_pred` + features. Do **not** block on estimating model θ or ICC `d_icc`. Temp/token tweaks won’t create useful ability disparity.
- **Why / evidence:** ICC linked ρ ~0.06 vs hybrid ~0.64 (full EN recollect); see Conceptual learnings Q&A same day. Smoke-fit metrics on disk (~0.50) are not the performance ceiling.
- **Follow-up:** EN full-force on frozen v4 when we want `out/d_est_trog_en_metrics.json` restored to full-panel quality.

### 2026-08-07 — CAT initial-`d` prior (product stance)

- **What:** Treat hybrid `d_est` as the recommended **initial bank `d` prior** for new CAT items / translations (panel + calibrator + features), then overwrite with field IRT.
- **Why / evidence:** Better than arbitrary/midpoint initials; EN TROG ranking ρ≈0.6+ on anchors; still a screen (MAE~0.8–0.9). Full write-up in §5 Q&A same day.
- **Follow-up:** Wire into CAT/item-bank authoring when EN full-panel `d_est` is restored; keep “not ground truth” in operator docs.

### 2026-08-08 — Operationalize hybrid `d_est` prior

- **What:** `apply_d_est_prior.mjs` + `QA_SIM_D_EST_PRIOR` in `simChildConfig.ts`. Established finite bank `d` never overwritten; blank `d` ← panel hybrid `d_est`.
- **Why / evidence:** Product goal = initial CAT `d` from simulated children without human field test; field IRT still final. Smoke apply on current EN cache: preserved=80, filled=19 (`out/d_est_prior_report_trog_en.md`).
- **Follow-up:** GCS/Crowdin promotion remains manual copy from `out/item_bank_*_d_est_prior.csv`. EN full-force still improves `d_est` quality.

### 2026-08-08 — Gate prior fill (skip BROKEN / known_issues)

- **What:** `apply_d_est_prior.mjs` leaves blank `d` when screen `flag=BROKEN` or UID ∈ `known_issues.json`.
- **Why / evidence:** Avoid writing bad CAT priors for duck/despite/car/sunny. Re-run: preserved=80, **filled=15**, **skipped=4**.
- **Follow-up:** EN full-force on frozen v4 when ready.

### 2026-08-08 — Blank bank `d` inventory (EN TROG)

- **What:** 19 scored bank rows lack shipped `d` but **are administered** (`test_response`, have `p_human`): Bishop nouns 1–4 (near-ceiling warm-ups) + Yeatman 86–103 (newer corpus without IRT `d` yet).
- **Why / evidence:** Not “unadministered.” Priors are mostly easy (median `d_est`≈−2.1); only car/truck ≈0; sunny/however extreme (+6.9, broken). See `out/d_est_prior_report_trog_en.md`.
- **Follow-up:** After gated apply, promote only OK/CEILING (and reviewed HARD) blanks; never BROKEN.

### 2026-08-08 — Improvement path (force → refit)

- **What:** Skip tighter OK/CEILING-only gate and age≤8 prompt check for now. Do **EN full-force** on frozen v4 (`panel_grid.json`, 32-cell replay) then **refit** calibrator + `estimate_difficulty` + gated apply.
- **Why / evidence:** Highest leverage for `d_est` quality; items 4–5 are secondary. Force started: `out/logs/en_v4_full_force_replay.log` (metrics pending finish).
- **Follow-up:** Chronology metrics when force completes.

### 2026-08-08 — Traditional panel ICC on 19 blank-bank items

**Hypothesis:** Accumulated panel runs → Rasch-with-guessing `d_icc` could set bank-scale `d` for blanks without hybrid `d_est`.

**Change:** Re-ran `fit_icc_difficulty.mjs --task trog --lang en` (87 runs; θ ages 6/8/10/12/13). Blank slice: `out/blank_d_icc_vs_d_est_trog_en.md` (+ `.csv`).

**Metrics:** ρ_icc_cv vs bank `d` = **0.080**; hybrid `d_est` = **0.647**. Nouns hit `all_correct` (d=−8); sunny/despite/car boundary (+8). Linked column compressed (flat affine).

**Verdict:** **NO-GO** for CAT prior — keep hybrid `d_est` + BROKEN gate. ICC diagnostic only; do not promote into bank.

**Next:** Optional ICC refresh after EN full-force finishes (trial counts only; quality ceiling unchanged).

### 2026-08-08 — EN full-force frozen v4 + post pipeline

**Hypothesis:** Full 32-cell force on frozen v4 restores full-panel hybrid `d_est` (~0.64) and refreshes gated priors with Redivis-latest human rates (`--human-source=bench` ← `responses/v2` from `levante_data_latest`).

**Change:** Force replay finished (`panel_grid.json`, ~1.7h). Then: `analyze --human-source=bench` → `estimate_difficulty` → gated `apply_d_est_prior` → `fit_icc_difficulty`.

**Metrics:**

| Signal | Value |
|--------|-------|
| EN screen | B4/H11/C14; ρ difficulty **0.64**; MAE cal **0.07** |
| Hybrid `d_est` ρ_multivar / −p_pred / MAE | **0.637** / 0.463 / 0.843 |
| Prior apply | preserved=80, filled=15, skipped_BROKEN=4 |
| ICC ρ_cv | 0.080 (still NO-GO) |

Duck/despite still **BROKEN** on full panel (`p_vlm`≈0.23/0.22) — smoke lift did not hold at 32 cells. Car/sunny remain known-suppressed.

**Verdict:** **GO** on hybrid `d_est` restore + gated prior draft. Prompt freeze stays v4; do not chase duck with more wording.

**Next:** Manual GCS promotion from `out/item_bank_trog_en_d_est_prior.csv` when ready.

### 2026-08-08 — Blind VLM `d_est` vs human-trial `d` on blanks

**Hypothesis:** New-item-blind hybrid `d_est` (no that-UID child data) matches bank-scale difficulty from collected human IRT.

**Change:** Linked Redivis `trog_item_params` → bank `d` (`d_human_bank`); compared to post-force `d_est` on 19 blanks. Write-up: `out/blank_d_human_vs_d_est_trog_en.md`.

**Metrics:** Anchors — VLM vs bank ρ **0.637** vs human-link CV ρ **0.502**. Blanks with both (13) — Spearman(`d_est`,`d_human_bank`) **0.709**, MAE 0.64. Six blanks lack human IRT rows.

**Verdict:** Blind hybrid is **competitive** with human-derived bank-scale `d` (ranking agree; slightly better bank recovery on anchors). Still a prior, not final field IRT.

**Next:** GCS promotion when ready; keep BROKEN skip.

### 2026-08-08 — Consolidated findings report

- **What:** Wrote `out/REPORT_d_est_initial_prior.md` — headline evidence (ρ≈0.64 bank recovery; vs human-link 0.50; blank agreement 0.71), ops prior fill, rejected paths, usefulness pitch.
- **Why / evidence:** Pulls post-force metrics + blank human comparison into one artifact for stakeholders.
- **Follow-up:** Manual GCS promote when ready.

### 2026-08-08 — Theory of Mind (stories) `d_est` first pass

**Hypothesis:** Same prior pipeline works when bank `difficulty` is entirely blank if we seed anchors from human IRT params.

**Change:** `estimate_difficulty` + `apply_d_est_prior` gain `stories` cfg; seed+flip bench `theory-of-mind_item_params`; features = `z` only.

**Run:** `fit_bench_calibrator --task stories --version v2` → `analyze --task stories --human-source=bench` → estimate → apply.

**Metrics:** LOO ρ vs human-IRT scale **0.55**; −p_pred ceiling **0.63**; prior filled **23** / skipped BROKEN **2** / no screen match **6**. Panel still June-era (INADEQUATE spread). Report: `out/REPORT_stories_d_est.md`.

**Verdict:** **GO for draft priors / triage**; weaker than TROG bank recovery. Recollect panel before production promote.

**Next:** Optional Stories force; review BROKEN + unmatched bank UIDs.

### 2026-08-09 — ToM: bank blank ≠ no human difficulty

- **What:** Clarified value of Stories `d_est`. Sim/CAT bank `difficulty` is empty for all scored ToM items, but Redivis/bench `theory-of-mind_item_params.csv` already has human IRT for ~31 UIDs (seeded as fit anchors, flipped easiness→harder-higher).
- **Why / evidence:** VLM prior mainly maps panel → that human-IRT scale into the blank bank column (filled 23). LOO ρ ≈ 0.55; −p_pred ceiling ≈ 0.63. Not a win over “never measured kids” — kids were measured; bank just never carried `difficulty`.
- **Follow-up:** True new-item value = ToM UIDs **without** item_params, or after a healthier panel recollect. Prefer TROG-style blank-bank story when bank `d` is missing *and* human IRT is missing.

### 2026-08-09 — Vocab EN `d_est` prior fill

**Hypothesis:** Same gated prior path as TROG works for vocab blanks (`vocab__*` bank UIDs).

**Change:** Refresh bench cal + analyze + estimate; fix `apply_d_est_prior` to index `bank_uid` (vocab_word_* → vocab__*).

**Metrics:** ρ_multivar **0.616** / −p_pred **0.654** / MAE 1.41 (126 anchors). Apply: preserved **126**, filled **44**, skipped 0. Report: `out/REPORT_vocab_d_est.md`.

**Verdict:** **GO** for draft prior (ops). Ranking still pass-rate-led; review CEILING fills before promote.

**Next:** Blank-vs-human-params write-up.

### 2026-08-09 — Vocab blank VLM vs human-linked `d`

**Hypothesis:** On bank-blank vocab items, blind hybrid `d_est` agrees with human IRT linked onto bank scale (TROG-style check).

**Change:** Affine-link `−item_params` → bank `d` on 125 anchors; compare on 19 blanks with params. Plain-language rewrite of report for readability.

**Artifacts:** `out/blank_d_human_vs_d_est_vocab_en.{csv,md}`, `out/REPORT_vocab_d_est.md`.

**Metrics:** Human→bank LOO ρ ≈ **0.933**; blank Spearman(`d_est`, `d_human_bank`) ≈ **0.532** (n=19, MAE≈1.06). **25/44** blanks have no human IRT.

**Verdict:** Prefer human params when present; VLM prior mainly for no-params blanks (weaker blank agreement than TROG’s 0.71).

**Next:** GCS promote after CEILING review; prompt experiments separate.

### 2026-08-09 — Vocab prompt v1 smoke (age-conditional)

**Hypothesis:** Age-split + anti-stretch prompts reduce CEILING pile-up and ease AI-too-easy mismatches (turnstile/colony).

**Change:** Extract `cypress/support/agents/prompts/vocabPrompts.ts` (v1); wire `vocabVlmAgent`. Grid `panel_grid_vocab_prompt_eval.json` (12 cells). **Must `--live`** (no vocab asset replay).

**Run:** All 12 cells exit 0 (~2.5 h). Analyze filtered to smoke run-ids → `screen_vocab_en_prompt_v1_smoke.csv`. Restored ops `screen_vocab_en.csv` to pre-v1 baseline.

**Metrics:** CEILING **101 → 132**; OK **41 → 7**; analyze ρDiff **~0.60 → 0.43**. turnstile/colony/typewriter moved **to CEILING** (worse).

**Verdict:** **NO-GO** for full EN recollect. Report: `out/REPORT_vocab_prompt_v1_smoke.md`.

**Next:** Prompt v2 (age vocab limit + don’t force match).

### 2026-08-09 — Vocab prompt v2 smoke

**Hypothesis:** Stronger uncommon-word rule + explicit “typical N-year-old vocabulary” limit reduces CEILING vs v1/baseline.

**Change:** `vocabPrompts.ts` v2 (`buildSystemPromptYoung` / `Checklist` inject age line). Same 12-cell `--live` smoke.

**Metrics:** vs baseline CEILING **101 → 125**, OK **41 → 18**, ρDiff **~0.60 → 0.54**. vs v1: CEILING 132→125, OK 7→18, ρ 0.43→0.54. turnstile/typewriter recovered; colony still CEILING; aesthete overshot to BROKEN.

**Verdict:** **Better than v1, still NO-GO vs baseline.** Report: `out/REPORT_vocab_prompt_v2_smoke.md`.

**Next:** Ability spread on current models (v3 knows-word randomize + temp ladder).

### 2026-08-10 — Vocab v3 ability smoke (knows-word → random + temp ladder)

**Hypothesis:** On current 3.5/3.6 models only, `DIGIT YES|NO` + agent randomize on NO + temps [0.5,1.2] reduces CEILING and restores ranking without Gemini 2.5.

**Change:** `vocabPrompts.ts` v3 + `applyKnowsWordPolicy`; `run_panel` `temperatures[]`; grid `panel_grid_vocab_ability_v3.json` (12 cells, `--live`).

**Run:** Initial 11/12; retried `36flash_a11_t05` (runId stamp fix when `temperatures.length===1`). **claw** → `known_issues.json` (screen + bank UIDs). Analyze → `screen_vocab_en_prompt_v3_ability_smoke.csv` (**resp=12**).

**Metrics:** CEILING **107**, OK **32**, HARD **29**, BROKEN **2** (claw known + gesticulate), ρDiff **0.64**.

**Verdict:** **GO** for current-model path vs v1/v2. Report: `out/REPORT_vocab_prompt_v3_ability_smoke.md`.

**Next:** Adopt v3 as default grid + refresh `d_est`.

### 2026-08-10 — Adopt vocab v3 defaults + refresh `d_est` prior

**Change:** `panel_grid_vocab.json` → temps `[0.5,1.2]`, repeats `2`, en-US, v3 notes. Promoted v3 smoke → `screen_vocab_en.csv`. `estimate_difficulty` + `apply_d_est_prior`.

**Metrics:** ρ_multivar **0.653** / −p_pred **0.746** / MAE **1.30** (126 anchors). Apply: preserved **126**, filled **44**, skipped_blocked **0** (claw already has bank `d`).

**Artifacts:** `d_est_vocab_en.*`, `item_bank_vocab_en_d_est_prior.csv`, `out/REPORT_vocab_d_est.md`.

**Verdict:** **GO** — v3 is the research default for vocab EN panels / priors.

### 2026-08-10 — Research framing (not promote)

- **What:** Clarified that AI `d_est` work is for **research utility to human researchers**, not production bank promotion.
- **Why:** Promote/GCS/Crowdin is out of scope; judge by ρ / MAE / ranking vs humans and usefulness on blanks lacking human IRT.
- **Follow-up:** Optional denser panel / Stories recollect / research write-up — see §6 open questions.

### 2026-08-10/11 — Matrix Reasoning first `d_est` smoke

**Hypothesis:** Same research question as TROG — can VLM panel + hybrid `d_est` recover bank `difficulty`?

**Change:** Wired `TASKS.matrix` (`stimulusAlt` / `joinByItemId`), bench mapping, estimate + apply prior; grid `panel_grid_matrix_smoke.json`.

**Run:** 6-cell EN `--live` (2 models × ages 6/8/11); all exit 0. Then analyze → fit_bench v2 → analyze `--human-source=bench` → estimate → apply prior.

**Metrics:** Join **78/80** UIDs; screen B11/H19/C9/OK41; ρ(p_vlm, human) **0.26**; anchors **75**; ρ_multivar **0.196** vs −p_pred ceiling **0.282**; MAE **0.837**. Prior: preserved 75 / filled 3.

**Verdict:** **Pipeline GO / research signal NO-GO** on smoke — join works; ranking too weak to help researchers yet (bottleneck = VLM↔human, not hybrid).

**Artifacts:** `out/REPORT_matrix_d_est_smoke.md`, `d_est_matrix_en_*`, `screen_matrix_en.csv`.

**Next:** Denser panel or matrix prompts only if chasing better ρ; else clone wiring to mental rotation.

### 2026-08-10/11 — Stories EN force recollect + `d_est` refresh

**Hypothesis:** Force recollect on current models reduces non-response / tightens ToM `d_est` vs human IRT.

**Change:** `panel_grid_stories.json` `--live --lang en-US --force` (18 cells, all exit 0). Analyze `--run-id-re '35flashlite|36flash'` → fit_bench v2 → estimate → apply prior.

**Metrics:** Non-response **33% → 0%**; spread still INADEQUATE (C22); ρ(p_vlm, human) **0.62**; LOO ρ_multivar **0.353** (was 0.55); −p_pred ceiling **0.683** (was 0.63). Prior: filled 21 / skipped 4 / no match 6.

**Verdict:** **Ops GO** (non-response fixed); research still prefer **−p_pred ranking** over hybrid for ToM. Spread/ceiling remains the limiter.

**Artifacts:** `out/REPORT_stories_force_en.md`, `d_est_stories_en_*`, `screen_stories_en.csv`.

### 2026-08-11 — Stories paper-ladder backtest

**Hypothesis:** Existing levante-bench open-model ToM runs can rank item difficulty vs human IRT (re-runnable alternative to Gemini panel).

**Run:** Aggregate `levante-bench/results/paper_models_forced_binary/*/theory-of-mind.csv` (17 models) → −p / ability-weighted −p vs flipped `item_params`; compare to Cypress force EN −p_pred. UID norm: strip `tom_storyN_`.

**Metrics:**

| Ladder | ρ(−p vs d_human) |
|--------|------------------|
| all-17 mean | 0.26 |
| all-17 ability-weighted | 0.43 |
| top-5 by ToM acc, weighted | **0.70** |
| Cypress −p_pred (force EN) | **0.69** |

**Verdict:** **GO for curated open ladder** (drop floor/broken 0–3% runs); full zoo too noisy. Competitive with live Gemini panel for Stories ranking.

**Artifacts:** `out/REPORT_stories_paper_ladder_backtest.md`, `out/backtest_stories_paper_ladder.csv`.

**Next:** Pin mid→strong open subset for new ToM items offline; optional same backtest for matrix/vocab.

### 2026-08-11 — Stories paper ladder → hybrid `d_est`

**Hypothesis:** Curated open ladder can drive the same `estimate_difficulty` path as Cypress.

**Run:** Top-5 ToM models → ability-weighted `p_vlm` → calibrate to bench pass-rates → `estimate_difficulty --task stories` (`screen_stories_en_paper_ladder.csv`).

**Metrics:** LOO ρ_multivar **0.612** / −p_pred **0.711** vs Cypress force **0.353** / **0.683**; paper vs Cypress `d_est_cv` ρ ≈ **0.78**.

**Verdict:** **GO** — ladder improves Stories `d_est` recovery vs CEILING-heavy Gemini panel.

**Artifacts:** `out/REPORT_stories_d_est_paper_ladder.md`, `d_est_stories_en_paper_ladder.csv`.

### 2026-08-11 — Literature: expert initial difficulty guesses

- **What:** Psychometric lit on SME / logical difficulty vs empirical *p* or IRT *b* (no LEVANTE expert-guess archive).
- **Why / evidence:** Single unaided absolute judgments often ρ ≈ **0.2–0.5**; pooled/trained/comparative methods often **~0.7–0.8+**. Ranking ≫ guessing % correct (Impara & Plake 1998; Bejar 1983; Attali/ETS; Wauters et al. 2011).
- **Follow-up:** Frame AI `d_est` (~0.64 TROG; ~0.70 Stories ladder) vs that human baseline in stakeholder notes — not vs perfect IRT.

### 2026-08-11 — Vocab paper-ladder backtest

**Hypothesis:** Same curated open-model ladder that worked for Stories can rank vocab difficulty vs human IRT / bank `d`.

**Run:** Aggregate `paper_models_forced_binary/*/vocab.csv` (15 models); UID `vocab__X` → `vocab_word_X`; compare −p / ability-weighted −p to flipped `vocab_item_params` and Cypress v3 `screen_vocab_en.csv`.

**Metrics:** Best paper subset ρ vs human ≈ **0.31** (all / mid); top-5-by-acc **worse** (~**0.17–0.20**, ceiling). Cypress −p_pred / −p_vlm ≈ **0.75 / 0.76**. vs bank `d`: paper ~**0.1** vs Cypress ~**0.53**.

**Verdict:** **NO-GO** — keep Approach A (Gemini v3 panel) for vocab. Stories “curate top-5” does not transfer when models are too strong.

**Artifacts:** `out/REPORT_vocab_paper_ladder_backtest.md`, `out/backtest_vocab_paper_ladder.csv`.

### 2026-08-11 — Vocab soft-score (YES|NO) retro

**Hypothesis:** Age-knows YES rate (`p_knows`) / soft expected hit (`p_soft`) differentiates ceiling items better than hard `p_vlm`.

**Run:** Retro on existing v3 12-cell panel (`analyze_vocab_soft.mjs`); no new live cells. Also log `knowsWord` / `modelIndex` / `randomized` on future vocab jsonl.

**Metrics:** −p vs bank `d`: `p_vlm` **0.577**, `p_knows`/`p_soft` **0.568**, `p_model` (pre-randomize) **0.313**. Ceiling ≥0.9: `p_vlm` 68% vs `p_knows` 66% (≈same). Pre-randomize digit is *more* ceilinged (91%).

**Verdict:** **NO-GO** — YES|NO soft score ≈ hard accuracy (v3 already folds NO→random into `p_vlm`). Next differentiation ideas: graded confidence, distractor confusability — not another binary soft layer.

**Artifacts:** `out/REPORT_vocab_soft_v3.md`, `out/screen_vocab_en_soft_v3.csv`, `analyze_vocab_soft.mjs`.

### 2026-08-11 — Vocab differentiation analysis (synthesis)

- **What:** For vocab ceiling, we checked (1) Cypress/Gemini v3 → hybrid `d_est`, (2) paper open-model ladder, (3) binary soft YES|NO on the same v3 logs.
- **Why / evidence:** Cypress still best: hybrid ρ≈**0.65** vs bank `d`, −`p_pred`≈**0.75**. Paper ladder best ≈**0.31** vs human (NO-GO). Soft `p_knows`≈`p_vlm` (**0.57**) — v3 already encodes knows→accuracy. Reports: `REPORT_vocab_d_est.md`, `REPORT_vocab_paper_ladder_backtest.md`, `REPORT_vocab_soft_v3.md`.
- **Follow-up:** Graded confidence smoke → **NO-GO** (2026-08-12); next ceiling probe = distractor confusability. Keep Approach A v3 panel for priors; do not expect Stories-style ladder curation to fix vocab.

### 2026-08-12 — Vocab graded confidence v4 smoke

**Hypothesis:** `DIGIT HIGH|MED|LOW` soft weights (1 / 0.5 / 0.25; LOW→random) differentiate ceiling items better than hard `p_vlm` / binary YES|NO.

**Change:** `QA_VOCAB_PROMPT=v4`; soft map in agent; `analyze_vocab_graded.mjs`.

**Run:** `panel_grid_vocab_graded_v4.json` live — **7/8** cells (1190 graded trials). `35flashlite_a11_t12` FAILED (screenshot timeout). Conf mix HIGH **0.69** / MED **0.09** / LOW **0.21**.

**Metrics:** −p vs bank `d`: `p_vlm` **0.592** > `p_knows` **0.566** > `p_graded` **0.530** (v3 ref −p_pred_child **0.533**). vs human: p_vlm **0.492** ≥ p_graded **0.471**. Ceiling ≥0.9: p_vlm **60%** → p_graded **54%** (small trim only). 1-cell peek LEAN-GO **reversed** on full pool.

**Verdict:** **NO-GO** — keep v3 default; do not wire graded into `d_est`.

**Artifacts:** `out/REPORT_vocab_graded_v4.md`, `out/screen_vocab_en_graded_v4.csv`.

**Next:** Distractor confusability probe (optional denser v3 panel stays optional).

### 2026-08-13 — SWR language fix + EN recollect vs human `b`

**Hypothesis:** Mis-labeled EN cells were German because `pickVariant` fell back to registered DE; injecting `lng`/`language` into Firekit params breaks `updateTaskParams`/startTask. Correct EN variant (name `en`, `language` null) + roar-swr `defaultToEnglish` should yield English items and enable human-`b` eval.

**Change:** Strict `pickVariant` (no silent DE fallback); do **not** inject `lng`/`language` into assessment params; `--run-suffix` on `run_roar_panel.mjs`; `eval_swr_b_est.mjs --en-suffix langfix`.

**Run:** `panel_grid_swr.json` EN-only `--run-suffix langfix` on local dashboard — **8/8** DONE (+ smoke `langfix7`). Bad prior EN/IT/PT dirs archived under `cypress/logs/runs/_archived_swr_langbug_20260812/`.

**Metrics:** EN→EN bank **471/471** (DE join 1); affine `b_est` vs human `b`: Spearman **ρ=0.175**, r=0.103, MAE=0.862 (n=471). DE ranking unchanged (`b` all 0).

**Verdict:** Language **GO**. Human-`b` alignment **weak** (ρ≈0.18) — ranking signal present but not ATM-ready calibration.

**Artifacts:** `out/REPORT_swr_en_de_b_est_eval.md`, `out/b_est_swr_en_eval.csv`, log `out/logs/swr_en_langfix_full.log`.

**Next:** Improve SWR difficulty signal (more respondents / better VLM policy) before seeding non-EN banks; retry ES; IT/PT need real variants.

### 2026-08-13 — SWR prompt v2 offline (child-conf)

**Hypothesis:** Adult LEFT/RIGHT play accuracy is the wrong construct; age-conditioned REAL|PSEUDO + HIGH|MED|LOW (child would judge correctly) should track human `b`.

**Change:** `cypress/support/agents/prompts/swrPrompts.ts` (`QA_SWR_PROMPT=v2`); wire `swrVlmAgent`; offline `eval_swr_prompt_v2.mjs`; `QA_PERSONA_AGE_YEARS` in `run_roar_panel.mjs`.

**Run:** text-only Gemini 3.5-flash-lite, age 8, n=40 from langfix words.

**Metrics:** lex vs bank **95%**; conf mix H/M/L **19/15/6**; ρ(p_child,b)=**−0.38**; ρ(b_proxy,b)=**0.41** (v1 live was ~0.18 and wrong-signed on p).

**Verdict:** **LEAN-GO** for live `QA_SWR_PROMPT=v2` smoke (still n=40 offline).

**Artifacts:** `out/REPORT_swr_prompt_v2_offline.md`, `out/swr_prompt_v2_offline.csv`.

**Next:** Live EN smoke with `QA_SWR_PROMPT=v2` (1–2 cells), then denser offline or full panel if ρ holds.

### 2026-08-14 — Literature: Zelikman et al. 2023 (SRE item gen + student simulator)

- **What:** Ingested EMNLP 2023 *Generating and Evaluating Tests for K-12 Students with Language Model Simulations* (SRE true/false + RT). Notes: `out/NOTES_zelikman_emnlp2023_sre.md`. Cite: https://aclanthology.org/2023.emnlp-main.135/
- **Why / evidence:** Two models — GPT-4 writes items from iterated expert rules; a **separate** LLM fine-tuned on student histories predicts unseen-item correctness + RT. Filter on simulated p/RT, then OT-match parallel forms. School n=234 grades 2–8: AI form vs lab **r=0.93**; crowdworker **r=0.92**. Simulator vs actual P(correct) **r=0.76**; vs item RT **r=0.50** (Flesch–Kincaid vs RT only **0.34**).
- **Follow-up:** Treat SRE/SWR child agents as **history-conditioned response simulators**, not the item-writer. Try expert-rule generation + simulator filter before panel `d_est`. See §7 Q&A.

### 2026-08-14 — SWR prompt bake-off → v3 live smoke

**Hypothesis:** HARDNESS 1–5 (vs H/M/L) and/or age 6 vs 10 would improve held-out ρ(b_proxy, human `b`).

**Change:** `eval_swr_prompt_bakeoff.mjs`; winner wired as `QA_SWR_PROMPT=v3` in `swrPrompts.ts` / `swrVlmAgent.ts` (text-only + hardness→`p_child`).

**Run:** Offline 120×4 conditions → winner `h15_age_avg` ρ≈**0.516**. Live: hung `v3smoke` (post-`xop`) archived; clean `v3smoke2` **DONE** exit 0 in 8.9 min (84 items).

**Metrics (live age-6):** lex **1.00**; hardness mix 2/3/31/25/23; ρ(b_proxy,b)=**−0.198** (v2 live was **+0.38**; offline h15_a6 ≈0.45).

**Verdict:** Plumbing **GO**; live metrics **NO-GO** on this cell (weak/inverted vs offline).

**Artifacts:** `out/REPORT_swr_prompt_bakeoff.md`, `out/REPORT_swr_prompt_v3_live_smoke.md`.

**Next:** Dual-age / age-avg live ensemble; compare ATM item draw vs bake-off sample before full panel.

### 2026-08-14 — Matched offline on `v3smoke2` words

**Hypothesis:** Live v3 ρ collapse is either (a) bad ATM item sample vs bake-off, or (b) live/ATM context corrupting ratings.

**Change:** `eval_swr_prompt_bakeoff.mjs --from-run panel_swr_en_35flashlite_a6_r1_v3smoke2` (84 words × hml/h15 × ages 6/10).

**Metrics:** ρ(live p, offline h15_a6 p)=**0.901** → ratings match. Offline **h15_a6** ρ(b_proxy,b)_all=**−0.13** (same failure as live). Best on this set: **h15_a10** ρ_test≈**0.29**; **hml_a10** / **hml_age_avg** ρ_all≈**0.29 / 0.27**. Original bake-off sample was optimistic.

**Verdict:** Not a live-pipeline bug — **this item set + age-6** is hard to rank. Prefer **v2** (or age-10 / dual-age) over age-6 v3.

**Artifacts:** `out/REPORT_swr_prompt_matched_offline.md`.

**Next:** Keep `QA_SWR_PROMPT=v2` for age-6 panels; optional age-10 v3 smoke only if chasing HARDNESS further.

### 2026-08-14 — Kuperman AoA baseline for SWR

**Hypothesis:** Age-of-acquisition norms can ground child vocabulary better than soft “typical N-year-old” prompt text.

**Change:** Copied `data/aoa_kuperman.csv`; `eval_swr_aoa_baseline.mjs` joins AoA / ortho / p_know to bank `b` and matched prompts.

**Metrics:** `langfix` reals n=221 ρ(AoA,b)=**0.498**; all-item p_know@6 ρ=**0.314**. Same ATM `v3smoke2` set: AoA reals ρ≈**0.07** (pseudos 0 hits); prompts still win there (hml_a10 ρ≈0.29).

**Verdict:** AoA is a **strong real-word feature** on the broad bank; useless alone on that ATM cell.

**Artifacts:** `out/REPORT_swr_aoa_baseline.md`.

### 2026-08-14 — AoA blend + prompt inject

**Hypothesis:** Kuperman AoA improves SWR `b` ranking via prompt inject and/or post-hoc blend with VLM `p_child`.

**Change:** `lib/aoa.mjs`, `eval_swr_aoa_blend.mjs`, `eval_swr_aoa_inject_offline.mjs`; `QA_SWR_AOA` + `lookupAoa` task; optional AoA line in `swrUserText`.

**Metrics:** Post-hoc **hml_a6** ρ 0.429→**0.452** (w=0.5). Prompt inject: a6 +0.007 / a10 **−0.070**.

**Verdict:** Use **post-hoc blend w≈0.5** for age-6 ranking; prompt inject **default off**.

**Artifacts:** `out/REPORT_swr_aoa_blend.md`, `out/REPORT_swr_aoa_inject_offline.md`.

### 2026-08-14 — Wire AoA blend into `eval_swr_b_est`

**Change:** Parse `pChild` from `modelRaw` when missing; plain logit; `--aoa-blend auto` (0.5 if graded panel else 0).

**Metrics:** `v2smoke4` ρ(b_proxy,b) **0.289 → 0.361** (+0.07). `langfix` auto→0 (no regression).

**Verdict:** **GO** — default ranking path for graded SWR panels.

**Artifacts:** `out/REPORT_swr_b_est_aoa_blend.md`.

### 2026-08-14 — SWR difficulty readiness (status)

**Hypothesis:** With EN language fixed, v2 child-conf, and AoA blend, are we close enough to estimate SWR item `b` for new/draft items?

**Stack (current best):** `QA_SWR_PROMPT=v2` (live) + `eval_swr_b_est.mjs --aoa-blend auto` (w=0.5 on graded panels). v3 HARDNESS offline-strong / live age-6 weak — not default.

**Metrics (vs human EN `b`):**

| Signal | ρ | n / note |
|--------|--:|----------|
| v1 play (`langfix`) | ~0.18 | 471 |
| v2 live `b_proxy` | **0.38** | 120 (`v2smoke4`) |
| v2 + AoA blend | **0.36** | 120 (plain-logit path; +0.07 vs unblended 0.29) |
| Offline bake-off HML+AoA w=0.5 | **0.45** | ~120 |
| Offline h15 age-avg (optimistic sample) | ~0.52 | held-out; did not transfer to ATM age-6 |
| Kuperman AoA alone (reals, `langfix`) | **0.50** | 221 reals; 0 for pseudos |

**Verdict:** **LEAN-GO for draft ranking** of EN items (ρ ~0.36–0.45 ≈ single-SME band). **NO-GO for calibrated ATM bank writes** / overwriting human `b`. Pseudos need VLM; reals benefit from AoA. Next: denser EN v2 panel for CI on ρ before any bank seed.

**Artifacts:** `out/REPORT_swr_prompt_v2_live_smoke.md`, `out/REPORT_swr_b_est_aoa_blend.md`, `out/REPORT_swr_aoa_baseline.md`.

### 2026-08-14 — Dual-age offline ensemble (in progress: full v2 panel)

**Hypothesis:** Averaging age-6 and age-10 HML `p_child`, then AoA-blend, beats single-age for ranking.

**Change:** `eval_swr_dual_age_ensemble.mjs`; live `QA_SWR_PROMPT=v2` full EN panel `--run-suffix v2full` (8 cells) started.

**Metrics (offline):** bakeoff age-avg ρ=**0.464** (+AoA **0.466**); fresh langfix n=120 age-avg **0.356** → +AoA **0.410**.

**Verdict:** Dual-age+AoA **GO** offline. Live denser panel pending (`v2full`).

**Artifacts:** `out/REPORT_swr_dual_age_ensemble.md`; log `out/logs/swr_en_v2full_panel.log`.

### 2026-08-15 — Full EN v2 panel (`v2full`)

**Run:** 8 cells, `QA_SWR_PROMPT=v2`; **6/8 DONE**, 2 FAILED stuck on `xop`.

**Metrics:** age-10 ρ≈**0.27**; age-6 ≈0.05; age-avg+AoA ≈**0.20**; offline dual-age+AoA was 0.41.

**Verdict:** Live ATM weaker than offline; prefer age-10 / offline ensemble. Hang still blocks full grids.

**Artifacts:** `out/REPORT_swr_v2full_panel.md`.

### 2026-08-15 — Fix SWR VLM post-answer hang (`xop`)

**Hypothesis:** Hang was oracle timing mismatch: VLM answered on first flash tick with multi-second polls/screenshots; timed 350ms trials stall, then false `block_transition` spam (~15 min gap).

**Change:** `cypress/e2e/swr/vlm_agent.cy.ts` — oracle-aligned loop (`seenTrialKey` / `lastAnsweredKey`, `isSwrAnswerableTrial`, 40ms poll, break debounce); skip viewport screenshot for v2/v3 when DOM word known.

**Run:** `panel_grid_swr_hangfix.json` → `panel_swr_en_35flashlite_a10_r1_hangfix1` (`QA_SWR_PROMPT=v2`).

**Metrics:** **84/84** items, 5 breaks, max gap 73s (tutorial→first); `presentationTime=350` throughout timed blocks; exit 0 in 5.6 min. (Prior failed cells: 1 item then ~924s gap + break spam.)

**Verdict:** Hang **fixed**. Safe to retry failed `v2full` cells / denser grids.

**Next:** Re-run failed `v2full` cells (or full panel with new suffix) and re-score ρ.

### 2026-08-15 — `v2full` retry complete (8/8)

**Change:** Archived hung cells → `_archived_swr_v2full_hung_20260815/`; re-ran 2 failed with hang-fixed agent.

**Metrics:** **8/8 × 84** items. age-10 ρ=**0.35** (was 0.27); pooled **0.28**; age-6 still weak (−0.09). AoA hurts live pools.

**Verdict:** Age-10 live **GO** for draft ranking; age-6-only NO-GO. Bank `b` writes still NO-GO.

**Artifacts:** `out/REPORT_swr_v2full_panel.md`; `out/logs/swr_en_v2full_retry.log`.

### 2026-08-15 — Draft EN ranking export

**Change:** `export_swr_draft_ranking.mjs` → age-10 live `v2full` (no AoA) + offline dual-age+AoA side-by-side.

**Metrics:** live a10 n=93 ρ=**0.348**; dual+AoA n=120 ρ=**0.389**; overlap 22 (different item sets).

**Verdict:** Draft rank lists ready. Prefer dual-age+AoA for general ranking; live a10 for ATM-seen words. Still NO-GO for bank `b` writes.

**Artifacts:** `out/swr_draft_rank_v2full_a10.csv`, `out/swr_draft_rank_dual_age_aoa.csv`, `out/swr_draft_rank_side_by_side.csv`, `out/REPORT_swr_draft_ranking.md`.

### 2026-08-15 — Break HIGH ceiling on EN bank (no ATM)

**Hypothesis:** Age-10 HML saturates (76% p=1 on dual-age fresh); bank-stratified offline + stricter HIGH anchors should raise ρ and cut ceiling.

**Change:** bakeoff `--source bank` + variant `hml_s`; `QA_SWR_PROMPT=v2strict` in `swrPrompts.ts`.

**Run:** 180 bank words × {hml, hml_s, h15} × ages 6/10 (1080 calls).

**Metrics:** age-10 HML ceil **59%→34%** (strict); **hml_s age-avg** ρ_all=**0.560** (ceil 5%) vs hml age-avg 0.447 (ceil 18%). Held-out ρ_test winner still `hml_a6` (0.42).

**Verdict:** Drop ATM for ranking. Prefer **`hml_s` age-avg** bank list (ρ≈0.56). Wire `v2strict` for future offline/live.

**Artifacts:** `out/REPORT_swr_prompt_bakeoff_bank.md`, `out/REPORT_swr_bank_ranking_ceiling.md`, `out/swr_draft_rank_bank_hml_s_ageavg_aoa.csv`.

### 2026-08-15 — Full EN bank `hml_s` age-avg ranking

**Run:** offline text-only `--source bank --limit 546 --variants hml_s --tag full` (no ATM/browser).

**Metrics:** n=**546**; age-avg ρ plain=**0.544** / +AoA **0.558**; ceil **5.1%**; held-out ρ_test age-avg **0.50**.

**Verdict:** Canonical draft ranking = full-bank `hml_s` age-avg (+AoA optional).

**Artifacts:** `out/swr_draft_rank_bank_hml_s_ageavg_full.csv`, `out/REPORT_swr_bank_ranking_full.md`, `out/swr_prompt_bakeoff_bank_full.csv`.

### 2026-08-15 — Status checkpoint (commit)

**Where we are:** Prefer **offline bank text-only** (`v2strict` / `hml_s` age-avg) over live ATM for EN difficulty ranking. Full bank n=546 reaches ρ≈**0.54–0.56** (Spearman — not chance; chance≈0). Live ATM still weaker (~0.35 age-10) and was hang-prone until oracle-aligned VLM loop.

**Still NO-GO:** writing calibrated ATM/`b` into the item bank.

**Go-from-here:** (1) held-out linear map `b_proxy→b` + residual audit (reals vs pseudos); (2) optional second-model confirm on bank; (3) only then draft-seed discussion.

### 2026-08-15 — Held-out `b_proxy→b` calibration

**Change:** `eval_swr_b_proxy_calibrate.mjs` on full-bank `hml_s` age-avg CSV.

**Metrics (test n=164):** AoA-blend ρ=**0.55**, MAE cal=**0.71** (raw MAE~5.6 — scale mismatch). Reals MAE 0.67 / ρ 0.52; pseudos MAE 0.76 / ρ **0.40**.

**Verdict:** **LEAN-GO** draft calibrated ranks; pseudos are the weak slice. Still review before bank writes.

**Artifacts:** `out/REPORT_swr_b_proxy_calibrate.md`.

### 2026-08-15 — PA ES VLM difficulty vs Bogotá human

**Hypothesis:** Text-only VLM can rank Spanish PA FSM/LSM items vs empirical human `b` from Bogotá trials; human n is already enough.

**Change:** `eval_pa_es_human_baseline.mjs`, `eval_pa_es_prompt_bakeoff.mjs`, `eval_pa_es_vlm_vs_human.mjs` (`lib/paEs.mjs`).

**Run:** 53 ES corpus items × {hml_s, h15} × ages 6/10 (212 calls, text-only).

**Metrics:** Human **53/53** items n≥394 (24.7k trials / 1397 runs) — **enough**. VLM vs human: h15 age-avg ρ=**0.485**; hml_s ρ=**0.458** (ceil 6%). LSM stronger than FSM.

**Verdict:** Human coverage **GO**. VLM ranking **LEAN-GO** (ρ≳0.4). No roar-pa CAT write.

**Artifacts:** `out/REPORT_pa_es_human_baseline.md`, `out/REPORT_pa_es_vlm_vs_human.md`, `out/pa_es_vlm_vs_human.csv`.

### 2026-08-15 — PA ES seed counterfactual (VLM `b` as bank params)

**Question:** If we had written Gemini `b` into ES PA, how would scoring / CAT have looked?

**Metrics (h15):** item MAE cal **0.57** (held-out 0.62); tertile same **43%** (3/53 easy↔hard). Observed-form θ ρ=**0.99** / MAE 0.26 (1,320 runs). CAT 19-item overlap vs human-`b` CAT: **42%** (chance 36%).

**Verdict:** Seed as **draft rank** only. **NO-GO** as CAT / published IRT `b` — adaptive forms barely beat random. Human Bogotá `b` is already estimable.

**Artifacts:** `out/REPORT_pa_es_vlm_seed_counterfactual.md`.

### 2026-08-15 — Ingest Ma et al. 2025 (BEA) inference RC AIG

- **What:** Read [arXiv:2506.08260](https://arxiv.org/abs/2506.08260) (Ma, Flor, Wang; BEA 2025). GPT-4o few-shot AIG of bridging-inference RC items.
- **Why / evidence:** 93.8% operational-quality items, but only 42.6% match the requested inference type. CoT did not help type control.
- **Follow-up:** Summary in [Appendix A](#appendix-a--ma-et-al-2025-bea-inference-rc-aig). Same lesson as our VLM `b` work: usable output ≠ construct lock.

### 2026-08-15 — Best task for generating new items

- **What:** Synthesis: which task to AIG first.
- **Why / evidence:** SRE has the only school-validated write+filter recipe (Zelikman r=0.93). Vocab wins *ranking* (−p_pred ~0.75), not writing (needs images). TROG/matrix are asset-heavy; matrix ranking NO-GO.
- **Follow-up:** Logged in §5. Still open: expert-rule GPT gen + simulator filter for SRE, then SWR.

### 2026-08-15 — Next ranking task

- **What:** Ranking-only: tackle **SWR EN pseudos** next (features / separate track).
- **Why / evidence:** Vocab/Stories/TROG already ~0.64–0.75. SWR `hml_s` 0.54–0.56; calibrate report says pseudos weaker; AoA is reals-only.
- **Follow-up:** Already an open checkbox. Do not chase matrix/PA-ES/SRE-VLM for ranking.

### 2026-08-15 — EGMA / mental rotation ranking probe

- **What:** Paper-ladder ρ vs human IRT (no new panel). EGMA |ρ|≈0.17–0.20 (n=116, ceiling). MR UID join broken (n=5). Matrix paper ρ≈0 confirms Cypress NO-GO.
- **Why / evidence:** `levante-bench/results/paper_models_forced_binary/*/{egma-math,mental-rotation,matrix-reasoning}.csv` + `irt_models/*_item_params.csv`.
- **Follow-up:** Do not queue ahead of SWR. Optional later: EGMA Cypress + type features; MR only after a UID map and only if matrix ranking moves.

### 2026-08-15 — SWR EN separate-track pseudos

**Hypothesis:** Pseudos lack AoA; length + real-bank bigram LL + neighborhood can lift ranking if fit on a separate track.

**Change:** `eval_swr_pseudo_features.mjs` (same hash split as calibrate).

**Run:** `node tools/vlm-panel/eval_swr_pseudo_features.mjs`

**Metrics:** held-out n=164. All ρ 0.555 → **0.589**. Pseudo n=84 ρ 0.395 → **0.507** (Δ+0.11). Real stays ~0.52–0.65 (AoA track). Report: `out/REPORT_swr_pseudo_features.md`.

**Verdict:** **LEAN-GO** for draft ranks. Still NO-GO for bank `b` writes.

**Next:** Optional: put guessed difficulties (`b_hat`) on the draft ranking sheet; do not chase more spelling features unless that export needs them.

### 2026-08-15 — Plain-language notebook pass

- **What:** Start-here story + glossary on the title page; plain intros on framing, results, learnings, and open questions. Dated experiment log kept.
- **Why:** Mixed audience. Chat rule `.cursor/rules/plain-english.mdc` now requires the same in replies.
- **Follow-up:** New chronology entries should lead with everyday words, then the symbol.

### 2026-08-17 — Default human trials: pilots v3.0

- **What:** Human kid-trial default is now Redivis **levante-data-pilots v3.0** (3,031 kids, 11 Aug) → local `data/responses/v3/`. Unified **levante-data-latest** stays an option (`--dataset levante_data_latest:e9pf:v1_2` → local `v2`) for when that bind is newer. QA `benchHuman` / calibrator / PA paths default to `v3`.
- **Why / evidence:** Theresa (Slack #levante-data-pipeline): pilots v3.0 newer than latest (20 Jul, 3,804 kids). Pins: `levante-bench/scripts/data_prep/download_levante_data.R`, `tools/vlm-panel/benchHuman.mjs`.
- **Follow-up:** Download v3 (Redivis OAuth), then `fit_bench_calibrator.mjs` so `--human-source=bench` pass-rates match the new tree. Do not overwrite local `v2`.

### 2026-08-17 — Split defaults: latest for QA, pilots v3 for public bench

- **What:** Internal QA keeps **levante-data-latest** (local `v2`). Public research (levante-bench) keeps **pilots v3.0** (local `v3`). Reverted `benchHuman` / calibrator / PA paths to `v2`.
- **Why / evidence:** Latest is the freshest bind for day-to-day QA. Pilots v3.0 is the latest public release, so it is the right default for published bench work.
- **Follow-up:** Bench still needs a Redivis login to fill `data/responses/v3/`. QA calibrators stay on the existing `v2` tree.

### 2026-08-23 — Vocab report: AI vs judges vs observed kids

- **What:** New write-up `out/REPORT_vocab_ai_vs_judges_and_observed.md` — vocab only. Primary question is how the panel ranks vs **human judges** (literature bands only; no Levante expert archive) and vs **observed** bank `d` / pass rates / IRT. Includes a methodology section (v3 panel → `p_vlm` → `p_pred` → hybrid `d_est`).
- **Why / evidence:** Same numbers as `d_est_vocab_en_report.md` and `blank_d_human_vs_d_est_vocab_en.md` (ranking −p_vlm ρ **0.763**, hybrid **0.653**, kids-IRT link **0.93**). Pointers added on those two files.
- **Follow-up:** Optional Levante expert-guess study still open.

### 2026-08-23 — TROG AIG first batch (pictures + d_est priors)

**Hypothesis:** We can write new EN TROG items (passive / X-but-not-Y) with pictures and guess difficulty with frozen v4 + hybrid `d_est`.
**Change:** `aig_trog_specs.mjs`, `aig_trog_images.mjs`, `aig_trog_score.mjs`, `aig_trog_lib.mjs`.
**Run:** draft 12 → lock 12/12 (too loose) → illustrate 6 → score (3.5-lite only; 3.6-flash vision silent).
**Metrics:** mean `p_vlm` **0.58**; 4/6 `d_est` in same-construction bank band; 2 chance-floor items explode `d_est` (~8–10). No kid ρ.
**Verdict:** LEAN-GO as a write+filter workshop; **NO-GO** as bank `d`. Report: `out/REPORT_trog_aig_en.md`.
**Next:** fix target-slot shuffle; tighten type-lock; do not fill priors when `p_vlm` = 0.

### 2026-08-27 — Vocab `d_est` uses registered locale CAT banks

- **What:** Fit/score English vocab guesses against `vocab-item-bank-en-US.csv` (CAT rule: `d` if present, else `difficulty`, skip ±5). Other langs map to `vocab-item-bank-<locale>.csv`. Helper: `vocab_bank_d.mjs`.
- **Why / evidence:** Registered `-prod` English (North America) uses the en-US file, not generic `vocab-item-bank.csv`. Anchors **144**. Ranking `−p_pred` **0.736**; hybrid ρ **0.604**, MAE **1.15**. Was 126 / 0.746 / 0.653 / 1.30 on `d` only.
- **Follow-up:** Same locale files. Fit later moved to `difficulty` (see next entries). Report: `out/REPORT_vocab_en_us_referee.md`.

### 2026-08-27 — Vocab `d` is old CAT; `difficulty` is kids IRT

- **What:** On `vocab-item-bank-en-US.csv`, `difficulty` (not ±5) equals the bench IRT with the sign flipped so higher = harder (143 words, match to 4 decimals). `d` is an older CAT column (125 words): same order (ρ ≈ 0.94) but ~0.94 units easier, never equal. Live CAT still prefers `d` (`getCorpus.ts`: `row.d` if present, else `row.difficulty`).
- **Why / evidence:** `difficulty` = `difficulty_registry`; ±5 live only in `difficulty_inferred`. If `difficulty` is truth, old `d` beats our CAT-fit `d_est` (overlap n=123: ρ **0.93** / MAE **1.07** vs held-out `d_est` **0.60** / **1.30**). Other shared banks are not two-layer: TROG is `d` only (not today’s IRT); matrix/math/mental-rotation ship `difficulty` only; Stories columns are empty.
- **Follow-up:** Retrain hybrid to `difficulty`. Do not replace live `d`.

### 2026-08-27 — Retrain vocab hybrid to kids IRT (`difficulty`)

**Hypothesis:** Fitting `d_est` to `difficulty` (not leftover CAT `d`) should recover kids IRT better.
**Change:** `vocabIrtD()` in `vocab_bank_d.mjs`; `estimate_difficulty.mjs` vocab load uses it. Apply-prior still `catVocabD` (what CAT treats as established).
**Run:** `node tools/vlm-panel/estimate_difficulty.mjs --task vocab --lang en`
**Metrics:** Anchors **143**. Held-out hybrid ρ **0.621**, MAE **1.08**, ranking `−p_pred` **0.740** (was 144 / 0.604 / 1.15 / 0.736 on the CAT rule). On 123 words with both numbers, old `d` still wins the sort (ρ **0.93** vs **0.61**); MAE almost tied (1.07 vs 1.10).
**Verdict:** LEAN-GO as the research referee. **NO-GO** as a replacement for live `d`.
**Next:** Use IRT-scale `d_est` only for words with no real `difficulty` yet.

### 2026-08-27 — Vocab pairwise “which is harder?” (text-only)

**Hypothesis:** YES|NO cannot split CEILING words (panel ~100% right). Asking which of two words is harder might recover kids IRT order on that subset.
**Change:** `eval_vocab_pairwise.mjs` (words only, no 4-ups). Age 8, `gemini-3.5-flash-lite`, k=4, 367 pairs.
**Run:** `node tools/vlm-panel/eval_vocab_pairwise.mjs --concurrency 6`
**Metrics:** Pair acc **0.665** all / **0.642** ceiling–ceiling / **0.850** vs HARD/OK (0.5 = coin flip). ρ(`bt`, IRT) **0.415** on 84 CEILING words vs `−p_pred` **0.182** and `d_est_cv` **−0.11**. All-pairs ρ **0.614** (n=114).
**Verdict:** LEAN-GO as a ceiling-breaker. Does **not** beat overall YES|NO ranking (~0.74). Still text-only — pictures not tried.
**Next:** Optional: pairwise with 4-ups, or fold `bt` in only for CEILING words. Report: `out/REPORT_vocab_pairwise_en.md`.

### 2026-08-27 — Vocab option (b): Kuperman AoA vs Zipf

**Hypothesis:** Adult-text Zipf is too flat among easy picture words. Age of acquisition (when people say they learned the word) should sort CEILING items and the no-IRT blanks.
**Change:** `eval_vocab_aoa.mjs` (offline join to `data/aoa_kuperman.csv`; no new panel).
**Run:** `node tools/vlm-panel/eval_vocab_aoa.mjs`
**Metrics:** Coverage **165/170**. AoA vs kids IRT ρ **0.720** (n=137) vs Zipf **0.338** and `−p_pred` **0.735**. On CEILING: AoA **0.400** vs Zipf **0.170** / `−p_pred` **0.182**. All **26** no-IRT words have AoA (duck 3.5 → divan 11.4) while `d_est` sits at ≈ **−1.5**.
**Verdict:** LEAN-GO as the sort key for easy / blank words. Do **not** replace the overall YES|NO ranking (tied with AoA). Report: `out/REPORT_vocab_aoa_en.md`.
**Next:** Optional: use AoA (not Zipf) when filling CEILING/blank priors.

### 2026-08-27 — Decision: re-fit vocab `d_est` features, not the panel

- **What:** AoA is a better feature than Zipf, but that does **not** mean re-running the VLM panel or throwing out `p_pred`. Re-fit the hybrid (`z` + **AoA** instead of Zipf) on the same `screen_vocab_en.csv`. Cheap; no new model calls.
- **Why / evidence:** Hybrid ranking already loses to `−p_pred` (ρ **0.62** vs **0.74**). Zipf is the weak extra feature (ρ **0.34** vs AoA **0.72**). Blanks collapse because Zipf is median-filled (3.41) while AoA spreads 3.5–11.4. Same panel stretch still owns the words the AI actually misses.
- **Follow-up:** Re-fit when we want better blank/CEILING priors. Do not overwrite live CAT `d`. Do not expect the new hybrid to beat `−p_pred` on the full list.

### 2026-08-27 — Re-fit vocab `d_est`: AoA instead of Zipf

**Hypothesis:** Same panel, swap Zipf → Kuperman AoA in the hybrid.
**Change:** `estimate_difficulty.mjs` vocab features `z` + `aoa` (rare/Zipf dropped from the fit; Zipf still in the CSV).
**Run:** `node tools/vlm-panel/estimate_difficulty.mjs --task vocab --lang en`
**Metrics:** Anchors **143**. Held-out hybrid ρ **0.749**, MAE **0.99** (was Zipf 0.621 / 1.08). Now slightly above `−p_pred` **0.740**. CEILING ρ **0.381**. 26 no-IRT `d_est` now spread **−2.35** (duck) to **1.81** (divan).
**Verdict:** LEAN-GO for research priors. Still **NO-GO** as live CAT `d`.
**Next:** Do not overwrite official `d`. Report: `out/d_est_vocab_en_report.md`.

### 2026-08-27 — Reports synced to AoA hybrid (no live `d` overwrite)

- **What:** Updated `out/d_est_vocab_en_report.md` (policy footer), `REPORT_vocab_en_us_referee.md` (AoA table), `REPORT_vocab_aoa_en.md` (re-fit pointer), `REPORT_vocab_d_est.md` and `REPORT_vocab_ai_vs_judges_and_observed.md` (current-number pointers). Generator `estimate_difficulty.mjs` now writes the same “do not overwrite live CAT `d`” line.
- **Why / evidence:** User: of course we do not replace official `d`. Docs had mixed Zipf-era ρ **0.62**.
- **Follow-up:** None. LEAN-GO research prior; NO-GO production `d`.

### 2026-08-28 — Meeting write-up refreshed to current method

- **What:** Rewrote `out/REPORT_vocab_ai_vs_judges_and_observed.md` so the body matches today’s method (kids IRT referee, `z`+AoA hybrid, do not overwrite leftover CAT `d`). Left 9–10 August ops/blank reports as dated snapshots.
- **Why / evidence:** Pointers at the top were current; the body still said bank `d` on 126 words and Zipf hybrid ρ **0.653**.
- **Follow-up:** None. Canonical story is this write-up + `d_est_vocab_en_report.md`.

### 2026-08-28 — Vocab de-DE + es-CO 12-cell v3 smokes (live, parallel)

- **What:** Started two live v3 grids (`panel_grid_vocab_ability_v3.json`, `--live`, 12 cells each). Logs: `out/logs/vocab_de_v3_smoke.log`, `out/logs/vocab_es_v3_smoke.log`. `saveManifest` now writes only the cell it just finished so two panels do not clobber `manifest.json`.
- **Why / evidence:** Locale banks already have kids IRT (151 de / 155 es). No `screen_vocab_{de,es}.csv` yet. English panel transfers at ρ ~0.64.
- **Follow-up:** When both finish → `analyze.mjs --task vocab` per lang → `estimate_difficulty.mjs --task vocab --lang de|es`. Resume failed cells (no `--force`).

### 2026-08-28 — Vocab de/es v3 smokes complete; first `d_est`

- **What:** Both 12-cell live smokes finished (Spanish last cell needed extra retries after leftover Cypress Chrome + `ECONNRESET`). Analyze `--run-id-re` v3 de|es only. Fit `z`+AoA vs locale kids IRT.
- **Metrics:** de anchors **151**, held-out hybrid ρ **0.492**, −p_pred **0.381**, MAE **1.34**. es anchors **153**, ρ **0.575**, −p_pred **0.416**, MAE **1.40**. (EN was 0.749 / 0.740 / 0.99.) Screens: `out/screen_vocab_{de,es}.csv`. Reports: `out/d_est_vocab_{de,es}_report.md`.
- **Verdict:** LEAN-GO as a first locale prior; weaker than EN. English-panel transfer vs locale IRT (~0.64) still beats these local −p_pred ranks.
- **Follow-up:** Do not overwrite leftover de CAT `d`. AoA table is English-only (next note).

### 2026-08-28 — AoA rulers are English-only

- **What:** The hybrid’s extra feature is Kuperman 2012 “typical age people learned this word.” That CSV (`data/aoa_kuperman.csv`) is **English**. We do **not** have a German or Spanish (or Dutch) age-of-acquisition table in the lab.
- **Why / evidence:** de/es `estimate_difficulty` reports the same **154** lexicon hits and median fill **7.32** as English — lookup is the English lemma on `item_uid` (`vocab_word_acorn` → acorn), not *Eichel* / *bellota*. So “when English speakers learned oak-nut” is standing in for “when German or Colombian kids learned this picture-word.” SWR AoA blend has the same limit (reals only, English norms).
- **Follow-up:** Optional: add a locale AoA list (or drop AoA on non-EN and use `z` only). Do not read de/es hybrid ρ as a fair test of the EN AoA trick.

### 2026-08-29 — Vocab translation sieve (Ruler B, |`d_est` Δ| ≥ 3)

- **What:** Asked whether the de/es panels can name words that fare badly (or too well) in translation, and how many of those “future” kids-IRT shifts we would have caught **before** locale child data. Two rulers: **A** = kids IRT `difficulty` (locale minus English); **B** = our guessed difficulty (`d_est` locale minus English). Empty `difficulty` is missing, not 0 (*squash* has no kids IRT in EN/DE/ES).
- **Why / evidence:** Joined `out/review_xlang_vocab_{de,es}.csv` with locale banks and `out/d_est_vocab_{en,de,es}.csv`. A 25-point panel swing (`p_vlm`) is a wide sieve (74 DE / 55 ES flags; ~1 in 10 later match a 2-unit kids shift, right sign). Tightening to **Ruler B |Δ| ≥ 3** flags **7** words; **5** later HIT kids |Δ| ≥ 3 with the same sign. |`d_est` Δ| ≥ 4 flags **none** (left off the review table). German kids IRT is mostly “easier” (scale not tightly linked).
- **Follow-up:** Human-check the 7-word list (picture + audio). *squash* → *Kürbis* / *calabaza* is the cleanest AI-only BROKEN (no kids IRT). Optional locale AoA still open.


## 4. Key quantitative results

**How to read the numbers:** ρ (rho) = did our easy→hard list match the kids’ list? **0 = chance, 1 = perfect; ~0.7 is useful sorting.** MAE = average miss (for percent-correct, 0.06 means we are off by about 6 points). Prefer the **bold** “best so far” rows, not one-off smokes.

### 4.1 Child pass-rate prediction (TROG EN)

“Can we guess what fraction of kids get a grammar-picture item right?” Lower MAE is better. `p_pred` is the AI rate stretched toward children.

| Snapshot | MAE p_vlm | MAE p_pred |
|----------|-----------|------------|
| Pre full recollect (`trog_en_pred_baseline.json`) | 0.108 | 0.076 |
| Post full recollect (`trog_en_pred_after.json`) | 0.104 | **0.063** |
| After age-eval mix (`trog_en_pred_age_eval.json` full screen) | 0.106 | **0.059** |

### 4.2 Guessed official hardness (`d_est`) — TROG English

“Does our guessed hardness list match the official bank list?” ρ closer to 1 is better. MAE here is in difficulty *points* (about 0.8 = typical miss), not percent.

| Snapshot | ρ multivar | ρ −p_pred | MAE multivar |
|----------|------------|-----------|--------------|
| Baseline full (`d_est_trog_en_baseline_full.json`) | 0.532 | 0.284 | 0.886 |
| Post full recollect (lab 2026-08-06; was `d_est_trog_en_metrics.json`) | **0.637** | **0.471** | **0.822** |
| Smoke-only overwrite 2026-08-07 (v4.2 `r[12]` screen) | 0.499 | 0.234 | 0.915 |
| Pre-force refresh 2026-08-08 (mixed disk → `d_est_trog_en_metrics.json`) | **0.647** | **0.456** | **0.833** |
| Post EN full-force v4 2026-08-08 (`d_est_trog_en_metrics.json`) | **0.637** | **0.463** | **0.843** |

**Note:** Prefer full-panel / post-force rows for “how well hybrid works.” Gated prior apply (post-force): preserved=80, filled=15, skipped_BROKEN=4. New-item use: ranking/triage, not CAT ground truth (see §5 Q&A).

Vocab EN (`d_est_vocab_en_metrics.json`, **v3 panel** 2026-08-10; AoA hybrid 2026-08-27): fit to kids IRT `difficulty` with `z`+AoA — ρ_multivar ≈ **0.749**; −p_pred ≈ **0.740**; MAE ≈ **0.99** (n_anchors=143). Was Zipf hybrid 0.621 / 1.08. Was CAT-rule 0.604 / 0.736 / 1.15 (n=144) and generic-`d` 0.653 / 0.746 / 1.30 (n=126). Old CAT `d` still beats `d_est` at matching `difficulty` on the overlap (ρ **0.93** vs hybrid **0.75**). **Do not overwrite live `d`.** Screen: C107/OK32/ρ≈0.64. Claw in `known_issues.json`. Reports: `out/d_est_vocab_en_report.md`, `out/REPORT_vocab_en_us_referee.md`. Pairwise text-only (`REPORT_vocab_pairwise_en.md`): CEILING ρ(`bt`) **0.415** vs `−p_pred` **0.182**. Kuperman AoA (`REPORT_vocab_aoa_en.md`): vs IRT ρ **0.720** overall / **0.400** on CEILING (Zipf **0.338** / **0.170**); all 26 no-IRT words have AoA. **AoA table is English-only.**

Vocab de-DE / es-CO first 12-cell v3 smokes (2026-08-28; `d_est_vocab_{de,es}_report.md`): same `z`+English-lemma AoA vs **locale** kids IRT. de: anchors **151**, ρ_multivar **0.492**, −p_pred **0.381**, MAE **1.34**. es: anchors **153**, ρ **0.575**, −p_pred **0.416**, MAE **1.40**. English panel vs those same locale IRT lists was ~**0.64** (no new panel). Screens: `out/screen_vocab_{de,es}.csv`. **Do not overwrite leftover de CAT `d`.** es-CO has no `d` column (CAT already uses `difficulty`).

Vocab translation sieve (2026-08-29; `out/review_xlang_vocab_{de,es}.csv` + `d_est_vocab_{en,de,es}.csv`). **Ruler B** = guessed-difficulty swing vs English. Flags at |`d_est` Δ| ≥ 3: **6 German + 1 Spanish**; **5 / 7** later match kids IRT |Δ| ≥ 3, same sign (− = easier, + = harder):

| Word | Gloss | Guess | Kids | |
|------|-------|-------|------|--|
| couturier | *Schneider* | easier (−3.3) | easier (−6.8) | HIT |
| aversion | *Abneigung* | easier (−3.1) | easier (−5.0) | HIT |
| facade | *Fassade* | easier (−3.0) | easier (−5.0) | HIT |
| posterior | *hinten* | easier (−3.3) | easier (−3.6) | HIT |
| cloak | *capucha* | harder (+3.9) | harder (+5.4) | HIT |
| concentric | *konzentrisch* | easier (−3.1) | a bit easier (−0.9) | MISS |
| turbine | *Turbine* | easier (−3.0) | a bit easier (−1.2) | MISS |

Wide 25-point `p_vlm` sieve (not the review cut): 74 DE / 55 ES flags. Kids |Δ| ≥ 3 issues: 32 DE / 9 ES; panel right-sign catch 6 / 3. Empty `difficulty` ≠ 0.

Matrix EN smoke (`d_est_matrix_en_report.md`, 2026-08-11): 6-cell panel; join **78/80**; ρ(p_vlm, human) **0.26**; anchors **75**; ρ_multivar **0.196** / −p_pred **0.282** / MAE **0.837**. Prior: preserved 75 / filled 3. **Pipeline GO / research signal NO-GO.** Report: `out/REPORT_matrix_d_est_smoke.md`.

Stories EN force (`REPORT_stories_force_en.md`, 2026-08-11): non-response **0%** (was ~33%); ρ(p_vlm, human) **0.62**; LOO ρ_multivar **0.353** / −p_pred **0.683**. Prefer −p_pred ranking for ToM research.

Stories paper open-model ladder backtest (`REPORT_stories_paper_ladder_backtest.md`, 2026-08-11): all-17 ρ **0.26**; curated top-5 ability-weighted **0.70** ≈ Cypress −p_pred **0.69** (n≈27). **GO** curated ladder; drop floor runs.

Stories paper ladder → `d_est` (`REPORT_stories_d_est_paper_ladder.md`, 2026-08-11): LOO ρ_multivar **0.612** / −p_pred **0.711** (Cypress force: **0.353** / **0.683**); paper vs Cypress `d_est_cv` ρ ≈ **0.78**.

Literature baseline (SME initial difficulty): single absolute guesses often ρ ≈ **0.2–0.5**; pooled/trained ranking often **~0.7+** (see §5 Q&A 2026-08-11).

Vocab paper open-model ladder (`REPORT_vocab_paper_ladder_backtest.md`, 2026-08-11): best ρ vs human ≈ **0.31**; Cypress v3 −p_pred ≈ **0.75**. Top-5-by-acc **hurts** (ceiling). **NO-GO** vs Approach A panel.

Vocab soft-score YES|NO retro (`REPORT_vocab_soft_v3.md`, 2026-08-11): −p_knows **0.568** ≈ −p_vlm **0.577** vs bank `d`; ceiling mass almost unchanged. **NO-GO** (binary soft ≈ hard after v3 policy).

Vocab graded HIGH|MED|LOW v4 smoke (`REPORT_vocab_graded_v4.md`, 2026-08-12): 7/8 cells; −p_vlm **0.592** > −p_graded **0.530** vs bank `d`; ceiling 60%→54%. **NO-GO** (keep v3).

SWR EN langfix recollect (`REPORT_swr_en_de_b_est_eval.md`, 2026-08-13): EN→EN bank **471/471**; `b_est` vs human `b` Spearman **ρ=0.175** (n=471). Language fixed; calibration weak.

### 4.2b Word reading (SWR) — guessed vs kids’ hardness (`b`)

Same idea as 4.2, for real and made-up words. **separate-track** = real words use “age kids learn this word”; made-up words use AI score + spelling patterns.

| Method | ρ vs human `b` | n | Verdict |
|--------|---------------:|--:|---------|
| v1 play accuracy (`langfix`) | ~0.18 | 471 | weak |
| v2 child-conf live (`v2smoke4`) | **0.38** | 120 | LEAN-GO |
| v2 + AoA blend (`b_est --aoa-blend auto`) | **0.36** | 120 | GO ranking path |
| Offline HML + AoA w=0.5 (bake-off) | **0.45** | ~120 | best offline |
| Kuperman AoA reals only (`langfix`) | **0.50** | 221 | lexicon prior; no pseudos |
| v3 live age-6 (`v3smoke2`) | −0.20 | 84 | NO-GO live |
| **`hml_s` age-avg full bank** | **0.54–0.56** | **546** | GO draft ranking (`v2strict`; ceil~5%) |
| **separate-track + ortho (held-out)** | **0.589** all / **0.507** pseudo | 164 / 84 | LEAN-GO vs baseline 0.555 / 0.395 (`REPORT_swr_pseudo_features.md`) |
| Live `v2full` age-10 (8/8) | **0.35** | ~93 | weaker than bank offline |
| **PA ES h15 age-avg vs Bogotá `b_human`** | **0.485** | 53 | LEAN-GO draft ES PA ranking |
| PA ES VLM-seeded CAT vs human-`b` CAT | overlap **0.42** | 19-item / 53 | **NO-GO** as published/CAT `b` (chance 0.36) |

Do **not** overwrite calibrated EN bank `b`. Draft ranking / triage only. Spearman ρ≈0.5 ≠ chance (chance≈0).

### 4.3 Hardness from age curves (`d_icc`) — TROG English

Attempt to back out official hardness from “older AI does better.” **Failed** (ρ ≈ 0.08). Keep as a diagnostic only.

| Snapshot | ρ linked `d_icc_cv` | ρ raw `d_icc` | vs hybrid `d_est` | n_runs |
|----------|---------------------|---------------|-------------------|--------|
| 2026-08-07 (pre a12-heavy) | ~0.06 | ~0.21 | 0.637 | 79 |
| 2026-08-08 pre-force refresh | **0.080** | (see report) | **0.647** | **87** |
| 2026-08-08 post full-force | **0.080** | (see report) | **0.637** | (updated runs) |

θ grid includes 6→−2.01, 8→−0.91, 10→−0.44, **12→−0.22**, 13→−0.15. Blank-item slice: `out/blank_d_icc_vs_d_est_trog_en.md`. **NO-GO** as bank prior.

### 4.4 Age gradient (mini-grid a6 vs a13; grid now uses a12)

See age-gradients track for expected VLM `acc=` ranges and child vs panel caveat. Artifacts: `age_grad_baseline_pre.json`, `age_grad_after.json`.

## 5. Conceptual learnings

**In one page**

1. **Best guess of “will kids get this right?”** Let several AIs take the real item, then stretch their percent-correct toward children (`p_pred_child`).
2. **Do not fake a child with the official formula** if the item has no official hardness yet — the formula has nothing to chew on.
3. **TROG (grammar pictures):** AIs were too hard on grammar; a checklist helped, but using the adult checklist at every age made 6-year-old and 12-year-old AIs look the same.
4. **Vocab:** AIs are too good at rare words. Telling them to “act younger” barely helps.
5. **Our guessed official hardness (`d_est`) is a sort key**, not the number the adaptive test should treat as truth.
6. **Same percent correct ≠ same official hardness.** Matching items that kids pass at 70% is a rough sort, not a calibration.
7. **Asking the AI to “pretend you are 6” failed.** Giving younger vs older AIs different *task* instructions worked a bit better.
8. **Ops:** resume unfinished runs; only redo successes after a prompt change; always `en-US`.

Q&As below keep the original technical detail. Symbols are in the [glossary](#glossary-say-these-out-loud).

### Model tier as “ability” — Q&A (2026-08-06)

**Q:** Is it a good idea to rely on differences in model ability when we don’t really know much about either model?

**A:** As a kid-ability continuum: **no**. As an engineering spread knob: **yes, with limits**.

We don’t know *why* flash-lite misses what flash gets. It isn’t θ — it’s a different training mix, size, and failure modes. Treating “lite → flash” like “age 6 → 12” overclaims the science.

What we *do* know empirically (EN TROG 3.5-flash-lite vs 3.6-flash):

- They **differ in score** a lot (~0.83 vs ~0.97) — enough to keep the spread gate alive; lite-only respondent SD ≈ 0.03 is too flat.
- That spread is mostly useful for **ranking / `p_vlm`**, which still tracks kid pass rates (ρ difficulty ~0.65, MAE `p_pred` ~0.08).
- It is **not** useful for kid-like discrimination (rpb ρ was negative). Different models don’t separate items the way different children do.
- Blind spots (negation, spatial, …) can be **shared** across tiers — then “ability” variance doesn’t rescue you; **cross-language shift** is the better check.

**Stance:** keep two models for spread; don’t interpret lite/flash gaps as developmental. Prefer claims validated against humans (`p_pred`, ρ difficulty, BROKEN catch) and xlang deltas over “the stronger model got it so the item is easy.” If cutting cost, drop repeats or an age before dropping a model.

### Accept hybrid `d_est` for new items — Q&A (2026-08-07)

**Q:** Can we estimate θ for panel models and use that to calculate bank-scale `d` for new items? Would sampling params (tokens, temperature) create useful ability disparity?

**A:** θ-for-models is doable (MLE/EAP on bank-`d` anchors from each cell’s response vector) but **won’t unlock new-item `d`** until VLMs show kid-like `P(correct|ability)` curves. Today we *assign* child mean θ by age (`age_task_ability.json`) as a persona cue; `fit_icc_difficulty.mjs` already uses that grid and linked `d_icc` ρ stays ~**0.05–0.06**. Re-estimating model θ mostly relabels the same flat curves. Temperature / max tokens / thinking budget mainly add noise or don’t apply (TROG answers are digits); **model tier + age scaffolding** remain the real spread knobs.

**Accept hybrid instead:** for unscored items, use panel → `p_pred_child` → **hybrid `d_est`** (`estimate_difficulty.mjs`: logit `z` + TROG construction tags / vocab Zipf → bank `d`), not ICC-from-θ.

**How well (held-out anchors, not field-proven on brand-new items):**

| Signal | What you get |
|--------|----------------|
| EN TROG hybrid ρ vs bank `d` | **~0.64** after full recollect (best); ~0.50 if fit on v4.2 smoke-only screen — don’t use smoke metrics as the ceiling |
| Pass-rate alone (`−p_pred`) | ~0.28–0.47 (hybrid beats when tags help) |
| Absolute MAE on bank `d` | ~0.8–0.9 — triage / draft CAT, not final bank rows |
| Child pass-rate MAE `p_pred` | ~**0.06** post full EN recollect |
| Vocab ranking | `−p_pred` ceiling ~**0.66**; hybrid ≈0.61 (features help MAE more than order) |

**Stance:** hybrid is good enough to **screen and rank** new TROG items. Don’t ship as CAT ground truth without field calibration. Restore full-panel `d_est` with EN force on frozen v4 when we need the ~0.64 number back on disk (`out/d_est_trog_en_metrics.json` was overwritten by a smoke fit on 2026-08-07). Sampling knobs are not the path to better `d`.

### Hybrid `d_est` as CAT initial-`d` prior — Q&A (2026-08-07)

**Q:** Is panel → hybrid `d_est` a useful innovation for how initial `d` is set when testing a new item on a CAT task?

**A:** **Yes — as a better prior, not a replacement for field calibration.**

Status quo initials are often a guess, a band midpoint, or “looks like item X,” weakly tied to child performance. This pipeline gives, *before any child data*, (1) calibrated `p_pred_child` and (2) bank-scale rank via hybrid `d_est` from a panel that took the **real item UI**. EN TROG held-out ρ ≈ **0.6+** vs bank `d` is clearly better than chance for ordering a draft bank.

**Innovative when:** stimulus-faithful (same task kids see); maps onto the **deployed bank scale** so CAT can start nearer the right difficulty region; surfaces BROKEN / xlang deltas pre-launch.

**Not innovative / limits:** still a **screen** (MAE on `d` ~0.8–0.9; VLM blind spots can mis-rank); must be overwritten once real trials exist; vocab gains less from hybrid tags than from panel `p` alone.

**Practical CAT use:** set initial `d` ← hybrid `d_est` (optionally shrink toward bank mean if uncertain) → field/CAT → refit IRT. Real ops win over arbitrary initials for new items and translations — only if panel `d` is never treated as ground truth. Skip writing priors for `BROKEN` / `known_issues.json` UIDs (`apply_d_est_prior.mjs`).

### Is `d_est` VLM-only? — Q&A (2026-08-08)

**Q:** Are estimated `d` values calculated only from VLM inputs, with no reliance on human results?

**A:** **No — not purely VLM.** For a *new* item (no kids on that UID): panel `p_vlm` and tags/Zipf are VLM/lexicon-only, but (1) `p_vlm`→`p_pred_child` uses a calibrator fit on **other items’ human pass rates**, and (2) hybrid coeffs map `z`+features → bank scale using anchors that have **human IRT bank `d`**. That item’s children aren’t required; the mapping is. `p_human` on the screen is eval-only, not an input to that item’s `d_est`.

### Blank bank `d` rows — Q&A (2026-08-08)

**Q:** Why do ~19 EN TROG items lack bank `d`? Are they unused? Are we just writing zeros?

**A:** They **are administered** (`test_response` + human p). Blank `d` means never shipped in the calibrated bank table: Bishop nouns 1–4 (near-ceiling) and Yeatman 86–103 (newer). Priors are **not** mostly ~0 — median `d_est`≈−2.1 (easy); one near-zero (car/truck); sunny/however extreme broken. Value is for OK/CEILING blanks; BROKEN panel items must stay blank (gated apply).

### Traditional panel ICC for blank `d`? — Q&A (2026-08-08)

**Q:** Can we set blank-bank `d` the traditional way (ICC from accumulated panel runs) instead of hybrid `d_est`?

**A:** We can compute it (`fit_icc_difficulty.mjs` → `d_icc` / `d_icc_linked`), and all 19 blanks have plenty of trials (~81–87). **Do not use it as the CAT prior.** Linked recovery of bank `d` stays ~**0.08** Spearman vs hybrid ~**0.65**; affine link is nearly flat; ceiling/floor items pin at ±8. Keep hybrid `d_est` (+ BROKEN skip). Artifacts: `out/blank_d_icc_vs_d_est_trog_en.md`, `out/d_icc_trog_en_report.md`.

### Blind VLM `d_est` vs human-trial `d` on blanks — Q&A (2026-08-08)

**Q:** For blank-bank items, is new-item-blind hybrid `d_est` competitive with `d` from collected human trials?

**A:** **Yes, competitively.** Human path = Redivis IRT `item_params` flipped and affine-linked to bank scale (`d_human_bank`). On **anchors**, VLM recovers bank `d` at ρ ≈ **0.64** vs human-link CV ρ ≈ **0.50**. On **13 blanks** with both estimates, rankings agree (ρ ≈ **0.71**). “Blind” = no *that-item* human data; calibrator/hybrid still use other items. Six blanks have no human IRT row. Write-up: `out/blank_d_human_vs_d_est_trog_en.md`.

### ToM bank blank vs human IRT — Q&A (2026-08-09)

**Q:** For Stories/ToM we said the bank has no difficulties — so did the VLM `d_est` invent difficulty without any human measurement?

**A:** **No.** Two stores: (1) CAT/sim bank `sim-item-bank-theory-of-mind.csv` — `difficulty` blank for all scored rows (what sim/CAT would read). (2) Redivis/bench `theory-of-mind_item_params.csv` — human IRT for ~31 `tom_*` UIDs. Stories estimate **seeds anchors from (2)** (flip easiness→harder-higher), then writes `d_est` into blank bank `difficulty`. So we filled the **bank column**; we did not replace a world with zero human params. Pipeline value today = ops + triage + readiness for **new** ToM items lacking item_params. Stronger “blind prior” story remains TROG blanks (or future ToM items) where human IRT is also missing. Report: `out/REPORT_stories_d_est.md`.

### Research vs promote — Q&A (2026-08-10)

**Q:** Do we need to promote AI `d_est` into production banks for this work to matter?

**A:** **No.** The question is whether AI difficulties help **human researchers** (priors, ranking, triage). Judge by recovery vs human IRT and usefulness where human params are missing—not by GCS/Crowdin ship readiness.

### Matrix first smoke — Q&A (2026-08-10)

**Q:** Does wiring matrix into the TROG-style `d_est` pipeline mean AI difficulties are useful for matrix researchers?

**A:** **Not yet.** Join via `stimulusAlt`/`item_id` works (78/80 UIDs). On a 6-cell EN smoke, ρ(p_vlm, human) ≈ **0.26** and ρ_multivar vs bank `difficulty` ≈ **0.196** (−p_pred ceiling ≈ **0.282**). Bottleneck is weak VLM↔child ranking, not hybrid mapping. Treat as pipeline proof; don’t claim researcher utility until ranking improves. Report: `out/REPORT_matrix_d_est_smoke.md`.

### Paper open-model ladder for Stories — Q&A (2026-08-11)

**Q:** Can the levante-bench paper’s open-model zoo replace a Gemini Cypress panel for estimating ToM item difficulties?

**A:** **Not the full zoo — a curated ladder can.** All-17 mean −p only reaches ρ ≈ **0.26** vs flipped human IRT (floor/broken runs at 0–3% ToM acc add noise). Ability-weighting helps (~**0.43**). A **top-5-by-accuracy** subset with ability-weighted −p reaches **ρ ≈ 0.70**, matching Cypress force EN −p_pred (**~0.69**). Feeding that ladder into `estimate_difficulty` yields LOO ρ_multivar **~0.61** (vs Cypress panel **~0.35** on the same force screen era). Prefer pinned mid→strong open weights for reproducibility; don’t treat model size as child θ. Reports: `out/REPORT_stories_paper_ladder_backtest.md`, `out/REPORT_stories_d_est_paper_ladder.md`.

### Expert initial difficulty guesses (literature) — Q&A (2026-08-11)

**Q:** How good are humans’ *initial* difficulty guesses before child data, and how should that frame AI `d_est`?

**A:** Literature (not a LEVANTE expert-guess archive): SMEs are better at **ranking** items than guessing absolute *p* / IRT *b*. Single unaided absolute judgments often correlate only **~0.2–0.5** with empirical difficulty; pooled or trained judges / comparative methods often reach **~0.7–0.8+**. Our EN TROG hybrid `d_est` (~**0.64** vs bank `d`) and Stories curated paper-ladder ranking / `d_est` (~**0.70** / **0.61**) sit in the **pooled-expert** band and above many single-guess baselines. We do **not** yet have LEVANTE content-expert guess→later-`d` data; that would be a useful local baseline study. Classic threads: Bejar (1983); Impara & Plake (1998); Attali et al. / ETS comparative judgment; Wauters et al. (2011).

### Vocab paper ladder — Q&A (2026-08-11)

**Q:** Does the Stories curated open-model ladder transfer to vocab?

**A:** **No.** Same `paper_models_forced_binary` zoo on `vocab.csv` tops out around ρ ≈ **0.31** vs flipped human IRT (Cypress v3 −p_pred ≈ **0.75**). Top-5-by-accuracy is *worse* (~0.17–0.20) because strong models sit near ceiling and lose item discrimination. Mid/spread subsets stay ~0.30. Prefer the live Gemini **v3** panel for vocab priors. Report: `out/REPORT_vocab_paper_ladder_backtest.md`.

### Vocab soft YES|NO score — Q&A (2026-08-11)

**Q:** Can `p_knows` (YES rate) beat hard `p_vlm` for vocab difficulty ranking?

**A:** **Not on the v3 panel.** Retro on 12 cells: −p_knows ≈ **0.57** vs −p_vlm ≈ **0.58** vs bank `d`; ceiling fractions nearly identical. That is expected: v3 already maps NO→random, so hard accuracy largely *is* the knows bit. Pre-randomize digit accuracy is worse (more ceiling). Try graded confidence or distractor confusability next, not another binary soft layer. Report: `out/REPORT_vocab_soft_v3.md`.

### Vocab graded HIGH|MED|LOW — Q&A (2026-08-12)

**Q:** Does three-level confidence beat hard `p_vlm` for vocab difficulty ranking?

**A:** **No on this smoke.** 7/8-cell live v4 panel: −p_graded **0.530** < −p_vlm **0.592** vs bank `d`; vs human pass-rate graded is also slightly worse. Ceiling trims only ~6pp (60%→54%). MED is rare (~9%); age-11 cells stay HIGH-heavy. 1-cell peek LEAN-GO did not hold. Keep **v3**; next differentiation idea is distractor confusability. Report: `out/REPORT_vocab_graded_v4.md`.

### SRE / new-item agents — Zelikman et al. 2023 (2026-08-14)

**Q:** What should we copy for SRE (and SWR/other CAT) agents and new-item banks?

**A:** Split **writer** and **child**. GPT-4 + a frozen expert-rule list writes sentences (grade-appropriate vocab, 3–15 words, universal facts; false = flip 1–2 content words; iterate rules on failure cases). A **different** model, fine-tuned on real student item-response sequences, replays *old students* on the new item (correctness + RT). Keep items that match target difficulty/ambiguity; build parallel forms with constrained OT (match simulated RT to lab items, no true↔false mix, no semantic clones). Do not use the writer as the child, and do not use Flesch–Kincaid as the RT prior. Their school check (r=0.93 vs expert lab form) is the bar. Notes: `out/NOTES_zelikman_emnlp2023_sre.md`. Our IRT/`timed_child`/VLM panel are cheaper stand-ins until we train a bench-history simulator.

### SWR difficulty estimation — readiness (2026-08-14)

**Q:** Are we close enough to estimate difficulty for SWR items?

**A:** **Yes for draft EN ranking / triage; no for calibrated ATM `b` or bank overwrite.** Best live path is v2 child-conf (`ρ≈0.38`) plus Kuperman AoA blend on reals (`ρ≈0.36` on `v2smoke4`; offline blend up to ~0.45). That sits in the single-expert guess band (~0.2–0.5), far below Zelikman-style school validation. Adult LEFT/RIGHT play is the wrong construct (~0.18). HARDNESS v3 won offline on a lucky sample but failed live age-6. AoA covers reals only; pseudos stay VLM. Next gate: denser EN v2 panel before any seeding.

### SWR — bank offline beats ATM (2026-08-15)

**Q:** Should we keep estimating SWR `b` via live ATM panels?

**A:** **No for ranking.** Stratified / full EN bank text-only with `v2strict` (`hml_s`) age-avg reaches ρ≈**0.54–0.56** (n=546, ceil~5%). Live ATM age-10 topped out ~0.35 and burned time on hangs. “Offline” = Gemini text ratings of bank words (not Cypress); use headless Cypress only when you need game play. Spearman ρ≈0.5 is moderate agreement, not chance (chance≈0). Still NO-GO for writing bank `b` without held-out calibration.

**Q:** If we seed PA ES items with VLM `b`, is that good enough?

**A:** **For ranking / triage, maybe; for CAT params, no.** Bogotá already has n≥394/item. VLM h15 ρ≈0.49 vs human `b`, but CAT using VLM `b` vs human `b` overlaps only ~42% of a 19-item form (chance 36%). Scoring θ on a **fixed** form stays aligned (ρ≈0.99) because 1PL is mostly number-correct. Do not replace estimable human `b`.

### AIG of diagnostic RC items — Ma et al. 2025 (2026-08-15)

**Q:** Can an LLM write operational inference-making RC items on demand?

**A:** **Quality yes; construct targeting no.** GPT-4o few-shot (T=0) hit 93.8% expert-acceptable MC items (grades 3–12), but only 42.6% were the requested bridging type (pronominal / text-connecting / gap-filling). CoT did not fix type control; more few-shot passages helped quality slightly. Pool-level type mix still resembled their operational bank. Use AIG + human review, not unattended type-conditional generation. Details: [Appendix A](#appendix-a--ma-et-al-2025-bea-inference-rc-aig).

### Best task for generating new items (2026-08-15)

**Q:** Which Levante task are we best at *generating* new, high-quality items for?

**A:** **SRE, then SWR.** We have not run AIG ourselves; this is construct + literature + our screen, not a yield number.

| Task | Why for AIG | Our ranking screen (ρ vs human) |
|------|-------------|----------------------------------|
| **SRE** | Only school-validated AIG recipe (Zelikman 2023: crowd r=0.92, school r=0.93). Text-only true/false, frozen expert rules, filter on simulated p+RT. | Our SRE VLM screen is ceilinged (INADEQUATE) — use their simulator, not panel-as-child |
| **SWR** | Lexicon + phonotactic pseudos, not sentence writing. Next after SRE in open questions. | Bank `hml_s` **0.54–0.56** (n=546) — draft rank only |
| Vocab | Best *filter* once an item exists; generation needs images + distractors. | Cypress −p_pred **~0.75** |
| Stories | Good rank; Ma-style construct-lock risk (ToM type). | Curated ladder **~0.70** (n≈27) |
| TROG | 4-picture grammar is hard to write validly (known BROKEN). | Hybrid **~0.64** |
| Matrix | Visual generation + ranking both weak. | **~0.20** NO-GO |

Do not confuse **writing** items with **ranking** them. Vocab wins ranking; SRE wins generation. Still need human review (Ma: 93.8% quality ≠ 42.6% requested type).

### Next ranking task (2026-08-15)

**Q:** If we only do ranking (no AIG), which task next?

**A:** **SWR EN, specifically pseudos.** Vocab (−p_pred ~0.75), Stories ladder (~0.70), and TROG hybrid (~0.64) are already in the triage / pooled-SME band. SWR full-bank `hml_s` is **0.54–0.56** (n=546) with a named residual: held-out `b_proxy→b` is weaker on pseudos; AoA covers reals only. That is the only ranking track with labels, a working offline method, and a clear gap.

Skip for ranking-next: matrix (VLM↔human ~0.26 NO-GO), PA ES (human `b` already estimable; CAT overlap ~42%), SRE VLM screen (ceilinged), vocab denser panel (diminishing). Stories next is ops (pin the ladder), not ρ. SWR ES is locale expansion after the EN pseudo gap.

### Math / shape-rotation for ranking? (2026-08-15)

**Q:** Should EGMA math or mental rotation (shape-rotation) jump the ranking queue?

**A:** **No — not ahead of SWR EN pseudos.** Both have human IRT (`egma-math` n=348, `mental-rotation` n=81) and Cypress VLM agents, but neither has a `d_est` panel. Ad-hoc paper-ladder on `paper_models_forced_binary` (same zoo as Stories):

| Task | Join n | all-model ρ(−p, d) | Notes |
|------|--------:|-------------------:|-------|
| EGMA | 116/348 | **−0.17** (ability-w −0.20) | Top models 87–95% acc — ceiling, vocab-ladder failure mode |
| Matrix | 75 | **−0.03** (top-5 −0.12) | Matches Cypress smoke NO-GO (~0.20–0.28) |
| Mental rotation | 5 exact | n too small | Paper UIDs are `mrot_…_Rp-020`; IRT is `mrot_…` / `…-2`. Acc 0.40–0.72 |

Shape-rotation is the same visual-spatial family as matrix (notebook already: clone MR only after matrix ranking is settled). EGMA is the more interesting *later* CAT (symbolic + type features; Cypress might beat the paper ceiling the way vocab did), but it is a new track, not a residual on a 0.54 method. Stay on SWR pseudos.

### SWR separate-track pseudos (2026-08-15)

**Q:** Can ortho features fix the SWR pseudo ranking gap?

**A:** **Yes as a separate track.** Same split as calibrate (seed=42, test 30%): pseudo ρ 0.395 → **0.507**; all-item ρ 0.555 → **0.589**. Reals stay on AoA-blend `b_proxy`. Best univariate on pseudos: real-bank bigram LL (ρ≈0.41) and length (≈0.31). VLM `p_avg` alone is weak on pseudos (≈0.11). Draft ranks only; still NO-GO for bank `b`. Script: `eval_swr_pseudo_features.mjs`.

### Which kid-trial file for QA vs public bench? (2026-08-17)

**Q:** Should internal QA and levante-bench share the same Redivis default?

**A:** **No.** QA uses **levante-data-latest** (local `data/responses/v2/`) because that bind is refreshed more often. Public bench uses **levante-data-pilots v3.0** (local `v3`) because that is the latest public release. Same download script; different `--dataset` / `--version`.

### Vocab `d` vs `difficulty` — Q&A (2026-08-27)

**Q:** Is `difficulty` the first guess and `d` the calibrated-after-kids number?

**A:** **The other way around** on English (North America). `difficulty` (skip ±5) **is** the kids IRT (registry; exact sign-flip of bench `vocab_item_params.csv`). `d` is leftover CAT hardness: same order (ρ ≈ 0.94), ~0.94 easier, never equal. Live adaptive vocab still prefers `d`. Other shared banks are not like this (TROG = `d` only and not today’s IRT; matrix/math/rotation = `difficulty` only; Stories empty).

**Q:** If `difficulty` is truth, are our guesses better than old `d`?

**A:** **No.** On 123 words with both: old `d` ρ **0.93** / MAE **1.07**. After retraining the hybrid to `difficulty`, held-out `d_est` is ρ **0.61** / MAE **1.10**. Ranking from the AI (`−p_pred` ≈ 0.74) is the useful part — for **new** words, not for replacing `d`.

### Vocab pairwise vs YES|NO — Q&A (2026-08-27)

**Q:** Does “which word is harder?” beat YES|NO on words the panel already aces?

**A:** **Yes, on that subset.** Text-only pairwise (age 8, 367 pairs): on 84 CEILING words, Bradley–Terry `bt` vs kids IRT ρ **0.415** vs YES|NO `−p_pred` **0.182**. Pair acc 0.64 among ceiling–ceiling (0.5 = coin flip). It does **not** replace the overall 0.74 YES|NO sort. Pictures not tried. `out/REPORT_vocab_pairwise_en.md`.

### Vocab AoA vs Zipf — Q&A (2026-08-27)

**Q:** Can “age kids learn this word” (Kuperman AoA) fix the easy-word pile-up?

**A:** **Yes, as a sort key — not as a replacement for the whole panel.** AoA vs kids IRT ρ **0.720** (n=137), almost the same as `−p_pred` **0.735**, and much better than Zipf **0.338**. On CEILING words AoA is **0.400** vs Zipf **0.170**. All 26 words with no IRT have an AoA and they spread (duck 3.5 years → divan 11.4) while `d_est` is stuck near −1.5. `out/REPORT_vocab_aoa_en.md`.

### Redo vocab `d_est` after AoA? — Q&A (2026-08-27)

**Q:** Does the AoA result mean we should redo `d_est`?

**A:** **Re-fit done (same panel).** Features are now `z` + AoA. Held-out ρ **0.749** / MAE **0.99** (was Zipf 0.621 / 1.08), slightly above `−p_pred` **0.740**. Blanks spread. **Do not replace live CAT `d`** — leftover `d` still matches IRT better on the overlap (ρ **0.93**). `out/d_est_vocab_en_report.md`.

### Vocab de/es smokes + English-only AoA — Q&A (2026-08-28)

**Q:** Can we run the English vocab method on German and Colombian Spanish?

**A:** **Yes for the panel and the kids-IRT referee; not a fair copy of the AoA trick.** Locale banks already have their own `difficulty` (151 de / 155 es). 12-cell live v3 smokes: held-out hybrid ρ **0.49** (de) / **0.58** (es) vs EN **0.75**. The English panel still sorts those locale IRT lists better (~0.64) than the local stretched percent-correct ranks (0.38 / 0.42).

**Q:** Do we have “age kids learn this word” for German or Spanish?

**A:** **No.** Kuperman 2012 (`data/aoa_kuperman.csv`) is an **English** ruler. de/es fits show the same 154 hits and median 7.32 as EN because we look up the English lemma on `item_uid`, not the spoken German/Spanish word. SWR uses the same English table (reals only). Until we add a locale AoA list, treat AoA on de/es as a borrowed English prior, not “when kids here learned it.” `out/d_est_vocab_{de,es}_report.md`.

### Vocab translation flags — Q&A (2026-08-29)

**Q:** Can the analysis point out terms that fare really poorly (or too well) in translation?

**A:** **Yes, with two rulers that often disagree.** Kids IRT (Ruler A) already shows large locale shifts (DE 32 / ES 9 words at |Δ| ≥ 3). The 25-point panel swing is a wide net. **Ruler B** (only flag when guessed `d_est` moves by 3+) is the tight sieve: **7** words, **5** later HIT. Six German flags are “easier”; only *capucha* is “harder.” Most kids-IRT extremes were already in the banks; *squash* → *Kürbis* / *calabaza* is AI-only (BROKEN, no kids IRT). `out/review_xlang_vocab_{de,es}.csv`.

**Q:** If we had run the AI before locale child data, how many future issues would it have found?

**A:** **A handful if we insist on the right easier/harder sign; more if we only build a review list.** At kids |Δ| ≥ 3, the 25-point panel would have named 6/32 German and 3/9 Spanish with the right sign. Tightening to Ruler B |`d_est` Δ| ≥ 3 would have put **7** words on a list, **5** later confirmed. German “easier” pile is partly unlinked IRT scales — not 32 broken translations.


## 6. Open questions / next

**Still open, in plain English**

- [ ] Put the new SWR guessed difficulties (`b_hat`) on the draft ranking spreadsheet (still do **not** overwrite official word difficulties).
- [ ] Confirm the word list with a second AI model.
- [ ] Retry Spanish word reading; Italian/Portuguese need real word lists first.
- [ ] Optional extras: denser vocab panel, vocab “confusable pictures,” more matrix work only if we insist on puzzles, pin the stories AI ladder, expert-guess study.
- [x] **Vocab CAT referee** — locale banks; live CAT = `d` else `difficulty` (skip ±5).
- [x] **Vocab `d` vs `difficulty`** — `difficulty` = kids IRT; `d` = older CAT. Live test prefers `d`.
- [x] **Retrain vocab hybrid to `difficulty`** — held-out ρ **0.621** / MAE **1.08** / −p_pred **0.740**. Old `d` still wins the sort (ρ **0.93**). `out/REPORT_vocab_en_us_referee.md`. Do **not** overwrite live `d`.
- [x] **Vocab pairwise-a (text-only)** — CEILING ρ(`bt`) **0.415** vs `−p_pred` **0.182**. LEAN-GO ceiling-breaker; not a full-bank replacement. `out/REPORT_vocab_pairwise_en.md`.
- [ ] Optional: pairwise with the actual 4-ups, or use `bt` only on CEILING words.
- [x] **Vocab AoA (option b)** — Kuperman vs IRT ρ **0.720** / CEILING **0.400** (Zipf 0.34 / 0.17). All 26 no-IRT words covered. `out/REPORT_vocab_aoa_en.md`.
- [x] **Re-fit vocab `d_est`** — `z`+AoA. Held-out ρ **0.749** / MAE **0.99** (was Zipf 0.621 / 1.08). Blanks spread −2.35…1.81. Still no live `d` overwrite.
- [x] **Vocab de-DE + es-CO 12-cell v3 smokes** — both 12/12. Hybrid vs locale IRT ρ **0.49** / **0.58**. `out/d_est_vocab_{de,es}_report.md`. Do not overwrite leftover de `d`.
- [x] **Vocab translation sieve (Ruler B)** — |`d_est` Δ| ≥ 3 flags **7** words, **5** HIT kids ≥ 3 (same sign). `out/review_xlang_vocab_{de,es}.csv`.
- [ ] Human-check the 7-word Ruler B list (picture + audio); *squash* has no kids IRT.
- [ ] Optional: **locale AoA** (German / Spanish tables). Kuperman is English-only; de/es currently reuse the English lemma.
- [x] **TROG AIG first batch** — specs + pictures + frozen v4 `d_est` (`out/REPORT_trog_aig_en.md`). LEAN-GO workshop; NO-GO bank `d`.
- [ ] Tighten TROG AIG type-lock and picture “follow” poses; do not use `d_est` when panel `p_vlm` = 0.
- [ ] Longer-term: train a “replay old students on a new sentence” model (SRE paper), then try writing new sentence-reading items.
- [ ] For **public bench only:** download Redivis **levante-data-pilots v3.0** into levante-bench `data/responses/v3/` (needs a fresh Redivis login). Internal QA stays on **latest** / local `v2`; do not point `fit_bench_calibrator.mjs` at v3 by default.

**Done / technical checklist** (kept for operators)

- [x] **v4 smoke replay**
 (`panel_grid_trog_v4_smoke.json`, 16 cells) — duck → OK; despite → HARD.
- [x] Re-analyze smoke-only (`--run-id-re '…_r[12]$'`) — ρ≈0.57, MAE p_pred≈0.083.
- [x] **v4.1 re-smoke** — car/truck still 0/16; duck softened; **NO-GO**.
- [x] **v4.2 re-smoke** — car/truck **0.06** still BROKEN; duck 0.56; ρ 0.53. **NO-GO**. Stop head-noun rewrites.
- [x] Freeze = **v4 prompt** + car/truck in `known_issues.json`.
- [x] Vocab analyze + `d_est` refresh — ρ_multivar 0.613 / −p_pred 0.661 (stable).
- [x] **de/es TROG force** (`panel_grid_trog_xlang_limited.json`) — done; ρ≈0.62–0.64; es strong_delta=0; de 1 spatial candidate.
- [x] Spot-check `trog_abovebelow_square_below_star` DE vs EN — above/below foil; not a translation break.
- [x] Decision: **accept hybrid `d_est`** for new-item bank-scale ranking (not model-θ / ICC). Documented in §5.
- [x] **Operationalize prior:** `apply_d_est_prior.mjs` + `QA_SIM_D_EST_PRIOR` sim overlay (preserve established `d`).
- [x] Gate prior fill — skip BROKEN / known_issues (filled 15, skipped 4).
- [x] Document blank-`d` inventory + “d_est not VLM-only” (§5).
- [x] Panel ICC on 19 blanks — refreshed; **NO-GO** vs `d_est` (`out/blank_d_icc_vs_d_est_trog_en.md`).
- [x] **EN full-force** on frozen v4 (32/32) + analyze/bench + `estimate_difficulty` + gated apply — ρ_multivar **0.637**; filled 15 / skipped 4. Duck/despite still BROKEN on full panel.
- [x] **Stories / ToM** first `d_est` pass — wired estimate/apply; seeded human IRT anchors; filled 23 / skipped 2 (`out/REPORT_stories_d_est.md`).
- [x] Document ToM **bank blank ≠ no human IRT** (ops fill vs new-item blind; §5 Q&A 2026-08-09).
- [x] **Vocab gated prior fill** — UID join fix; preserved 126 / filled 44 (`out/REPORT_vocab_d_est.md`).
- [x] **Vocab blank vs human-linked `d`** — ρ ≈ 0.53 on 19; 25 no-params (`out/blank_d_human_vs_d_est_vocab_en.md`).
- [x] **Vocab prompt v1 smoke** — NO-GO (CEILING ↑); `out/REPORT_vocab_prompt_v1_smoke.md`.
- [x] **Vocab prompt v2 smoke** — better than v1, still NO-GO vs baseline; `out/REPORT_vocab_prompt_v2_smoke.md`.
- [x] **Vocab v3 ability smoke** — GO on current models (C107 / OK32 / ρ0.64; 12/12 after retry); claw in `known_issues.json`; `out/REPORT_vocab_prompt_v3_ability_smoke.md`.
- [x] **Adopt vocab v3 defaults** — `panel_grid_vocab.json` + refreshed `d_est` / prior (ρ_multivar **0.653**); `out/REPORT_vocab_d_est.md`.
- [x] **Matrix Reasoning first `d_est` smoke** — wired join/estimate; 6/6 cells; ρ_multivar **0.196** (pipeline GO / signal NO-GO); `out/REPORT_matrix_d_est_smoke.md`.
- [x] **Stories EN force recollect** (`panel_grid_stories.json`, `--live --lang en-US --force`) — non-response **0%**; −p_pred ceiling **0.683**; hybrid LOO ρ **0.353**; `out/REPORT_stories_force_en.md`.
- [x] **Stories paper-ladder backtest** — curated top-5 weighted ρ **0.70** ≈ Cypress **0.69**; `out/REPORT_stories_paper_ladder_backtest.md`.
- [x] **Stories paper ladder → `d_est`** — LOO ρ **0.612** vs Cypress **0.353**; `out/REPORT_stories_d_est_paper_ladder.md`.
- [x] **Literature baseline** — SME initial difficulty guesses typically ρ ~0.2–0.5 (single) / ~0.7+ (pooled); frame AI vs that (§5 Q&A).
- [x] **Vocab paper-ladder backtest** — **NO-GO** (best ρ≈0.31 vs Cypress≈0.75); top-5 hurts; `out/REPORT_vocab_paper_ladder_backtest.md`.
- [x] **Vocab soft-score (YES|NO) retro** — **NO-GO** (−p_knows≈0.57 ≈ −p_vlm≈0.58); `out/REPORT_vocab_soft_v3.md`.
- [x] **Vocab graded confidence v4 smoke** — **NO-GO** (7/8 cells; −p_vlm **0.592** > −p_graded **0.530**); `out/REPORT_vocab_graded_v4.md`.
- [ ] Optional denser vocab EN panel under default grid (`repeats: 2` → 24 cells) for stabler research estimates.
- [ ] Optional vocab **distractor confusability** probe (graded also NO-GO).
- [ ] Optional denser matrix panel / prompt work only if chasing better ρ (smoke bottleneck = VLM↔human ≈0.26).
- [ ] Optional Stories ability-spread work (temps / harder personas) — panel still CEILING-heavy after force; or prefer curated open ladder path.
- [ ] Pin curated open ToM ladder + offline new-item scoring path (follow paper-ladder backtest / `d_est`).
- [ ] Optional paper-ladder backtest for **matrix** (vocab done — NO-GO).
- [ ] Optional LEVANTE expert-guess study (content experts rate held-out items → compare to later bank/`item_params`).
- [x] Research write-up / Slack stakeholder note: AI `d_est` vs bank `d` vs typical human initial guesses (`out/SLACK_d_est_research_status.md`).
- [ ] Decide whether age ≤8 light prompt is too aggressive for age 8 (deferred; not needed for prior pipeline).
- [ ] Keep this notebook updated after each experiment.
- [x] **SWR language fix + EN recollect** — `pickVariant` strict; no Firekit `lng` inject; EN `langfix` 8/8. EN→EN **471/471**; vs human `b` ρ=**0.175** (`out/REPORT_swr_en_de_b_est_eval.md`).
- [x] **SWR prompt v2 offline** — child-conf REAL|PSEUDO+HIGH|MED|LOW; n=40 ρ(b_proxy,b)=**0.41** (`out/REPORT_swr_prompt_v2_offline.md`). **LEAN-GO** → live smoke next.
- [x] **SWR prompt bake-off** — HML vs HARDNESS 1–5 × ages 6/10; winner `h15_age_avg` held-out ρ≈**0.516** (`out/REPORT_swr_prompt_bakeoff.md`).
- [x] **SWR `QA_SWR_PROMPT=v3` live smoke** — `v3smoke2` DONE 84 items; lex 1.0; ρ(b_proxy,b)=**−0.20** (`out/REPORT_swr_prompt_v3_live_smoke.md`). Plumbing GO / metrics weak vs offline+v2 live.
- [x] **SWR matched offline diagnosis** — same 84 words; live↔offline p ρ=**0.90**; age-6 offline also fails; prefer **v2** / age-10 (`out/REPORT_swr_prompt_matched_offline.md`).
- [x] **SWR Kuperman AoA baseline** — `langfix` reals ρ(AoA,b)=**0.50**; ATM cell AoA≈0 (`out/REPORT_swr_aoa_baseline.md`).
- [x] **SWR AoA blend + inject** — post-hoc w=0.5 lifts hml_a6 0.429→**0.452**; prompt inject hurts a10 (−0.07) → inject default off (`out/REPORT_swr_aoa_blend.md`).
- [x] **SWR `b_est` AoA blend** — `auto` w=0.5 on graded panels; `v2smoke4` ρ **0.289→0.361** (`out/REPORT_swr_b_est_aoa_blend.md`). Keep live `QA_SWR_PROMPT=v2`.
- [x] **SWR readiness call** — LEAN-GO draft EN ranking (ρ~0.36–0.45); NO-GO calibrated bank `b` writes (`§5` 2026-08-14 status).
- [x] **SWR denser EN v2 panel (`v2full`)** — 6/8 DONE; age-10 ρ≈0.27; 2× `xop` hangs (`out/REPORT_swr_v2full_panel.md`).
- [x] **SWR fix post-answer `xop` hang** — oracle-aligned VLM loop; smoke `hangfix1` 84/84 @350ms (`§5` 2026-08-15).
- [x] **Break HIGH ceiling on bank** — `hml_s` age-avg ρ≈**0.56**, ceil 5% (`out/REPORT_swr_bank_ranking_ceiling.md`). Prefer bank offline over ATM.
- [x] **Full 546-word bank `hml_s` age-avg** — ρ≈**0.54–0.56**, ceil 5% (`out/swr_draft_rank_bank_hml_s_ageavg_full.csv`).
- [x] **Held-out calibrate** `b_proxy→b` — test MAE **0.71**, ρ **0.55**; pseudos weaker (`out/REPORT_swr_b_proxy_calibrate.md`).
- [x] **PA ES VLM vs human** — Bogotá n enough (53/53 ≥394); h15 ρ=**0.485** (`out/REPORT_pa_es_vlm_vs_human.md`).
- [x] **PA ES seed counterfactual** — CAT overlap ~42% (chance 36%); **NO-GO** as published/CAT `b` (`out/REPORT_pa_es_vlm_seed_counterfactual.md`).
- [x] Improve **pseudo** ranking (features / separate track) — held-out ρ 0.395 → **0.507**; all 0.555 → **0.589** (`out/REPORT_swr_pseudo_features.md`).
- [ ] Optional: wire separate-track `b_hat` into SWR draft rank export (still no bank `b` writes).
- [ ] Optional second-model confirm on bank (`gemini-3.6-flash` or similar).
- [ ] **SWR ES retry**; IT/PT blocked until real variants exist (no DE fallback).
- [ ] **SRE/SWR item-response simulator** on levante-bench histories (student-conditioned p + RT), vs IRT/`timed_child`/VLM panel — Zelikman et al. 2023 (`out/NOTES_zelikman_emnlp2023_sre.md`).
- [ ] **Expert-rule GPT item gen + simulator filter** for SRE (then SWR) new items, before panel `d_est`.
- [ ] **Parallel-form OT matching** if we need multiple SRE/SWR banks (don’t greedy-nearest-neighbor).

---

### Entry template (copy for new dated sections)

```markdown
### YYYY-MM-DD — <title>

**Hypothesis:** …
**Change:** … (files)
**Run:** … (grid / command)
**Metrics:** … (table + artifact paths)
**Verdict:** …
**Next:** …
```

## 7. Reload live artifacts

Run the cells below to refresh tables from `out/` without editing the narrative above.

In [ ]:
from pathlib import Path
import json

OUT = Path("out")
if not OUT.exists():
    OUT = Path("tools/vlm-panel/out")

def load(name):
    p = OUT / name
    if not p.exists():
        print(f"missing: {p}")
        return None
    return json.loads(p.read_text())

files = [
    "d_est_trog_en_baseline_full.json",
    "d_est_trog_en_metrics.json",
    "trog_en_pred_baseline.json",
    "trog_en_pred_after.json",
    "trog_en_pred_age_eval.json",
    "d_icc_trog_en_metrics.json",
    "age_grad_baseline_pre.json",
    "age_grad_after.json",
    "d_est_vocab_en_metrics.json",
]
data = {f: load(f) for f in files}
print("Loaded", sum(v is not None for v in data.values()), "/", len(files), "from", OUT.resolve())

In [ ]:
def fmt(x, d=3):
    return "—" if x is None else f"{x:.{d}f}"

base = data.get("d_est_trog_en_baseline_full.json") or {}
cur = data.get("d_est_trog_en_metrics.json") or {}
print("=== TROG d_est vs bank d ===")
print(f"{'metric':28} {'baseline':>10} {'current':>10} {'Δ':>10}")
for k in ("spearman_multivar", "spearman_neg_p_pred", "mae_multivar"):
    b, c = base.get(k), cur.get(k)
    delta = None if b is None or c is None else c - b
    print(f"{k:28} {fmt(b):>10} {fmt(c):>10} {fmt(delta):>10}")

pb = data.get("trog_en_pred_baseline.json") or {}
pa = data.get("trog_en_pred_after.json") or {}
print("\n=== TROG p_pred MAE ===")
print(f"baseline MAE p_pred: {fmt(pb.get('mae_p_pred_vs_human'))}")
print(f"after full recollect: {fmt(pa.get('mae_p_pred_vs_human'))}")

ag0 = data.get("age_grad_baseline_pre.json") or {}
ag1 = data.get("age_grad_after.json") or {}
print("\n=== Age gradient a6 vs a13 ===")
print(f"Δ med_p before: {fmt(ag0.get('delta_med_p'))}")
print(f"Δ med_p after:  {fmt(ag1.get('delta_med_p'))}")

icc = data.get("d_icc_trog_en_metrics.json") or {}
print("\n=== ICC ===")
print(f"ρ d_icc_cv: {fmt(icc.get('spearman_d_icc_cv'))}")
print(f"ρ −p_pred:  {fmt(icc.get('spearman_neg_p_pred'))}")

---

*End of initial lab dump (2026-08-07). Append below.*

## Appendix A — Ma et al. 2025 (BEA) inference RC AIG

**Paper:** Wanjing Anya Ma, Michael Flor, Zuowei Wang. *Automatic Generation of Inference Making Questions for Reading Comprehension Assessments.* BEA 2025 (ACL). [arXiv:2506.08260](https://arxiv.org/abs/2506.08260). Work done at ETS; Ma now at Stanford.

**Ingested:** 2026-08-15 from arXiv HTML.

### Claim

GPT-4o can draft **usable** diagnostic RC items at scale, but it does **not** reliably hit a requested inference *type*. Human review is required before operational use.

### Taxonomy (validated on an ETS bank)

Bridging inferences (local/global coherence) vs literal/vocab/other. Bridging = **51%** of 192 expert MC items on 24 expository passages (grades 3–12). Dual-coder type agreement 86%, κ=0.83.

| Type | What the reader must do |
|------|-------------------------|
| Pronominal | Direct pronoun resolution (“who is *he*?”) |
| Pronominal bridging | Pronoun as a hint that links sentences |
| Text-connecting | Causal/logical link stated across sentences |
| Gap-filling | Text + world knowledge to fill an unstated detail |

### Method

- Few-shot GPT-4o (`2024-04-01-preview`, T=0): 4 vs 6 example passages; standard vs CoT (text hint + reasoning).
- 10 new Simple Wikipedia expository passages; 3 types × up to 3 items; **357** items (180 CoT).
- Three expert raters: (1) operational quality, (2) requested type, (3) reasoning/hint adequacy.

### Results (Table 4)

| Condition | n | Quality | Type hit | Reasoning |
|-----------|--:|--------:|---------:|----------:|
| standard_4 | 88 | 0.932 | 0.409 | — |
| standard_6 | 89 | **0.955** | **0.461** | — |
| CoT_4 | 90 | 0.900 | 0.411 | 0.356 |
| CoT_6 | 90 | **0.967** | 0.422 | 0.389 |
| **Total** | **357** | **0.938** | **0.426** | **0.372** |

CoT did **not** improve type control. More examples helped quality. Generated *pool* type mix still matched the operational bank (Fig. 6) even when item-level targeting failed.

### Limits (authors)

10 expository passages only; GPT-4o only; CoT null result may be few-shot starved. No narrative genre.

### For this lab

Same split we keep hitting: **LLM output can look operational while missing the construct you asked for** (cf. VLM `b` ranking vs CAT seed). If we AIG stories/RC items, generate then human-type-check; do not treat “write a gap-filling item” as a locked spec.
